<a href="https://colab.research.google.com/github/Derinhelm/parser_stat/blob/mmd_metric/MMD_Russian_parser_statistics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Repository cloning

In [ ]:
!git clone https://github.com/Derinhelm/parser_stat.git
!cd parser_stat && git checkout tokenization_changing

fatal: destination path 'parser_stat' already exists and is not an empty directory.
Already on 'tokenization_changing'
Your branch is up to date with 'origin/tokenization_changing'.


In [ ]:
import sys
sys.path.append('/content/parser_stat')

In [ ]:
!mkdir pics

mkdir: cannot create directory ‘pics’: File exists


# Data getting

In [ ]:
from data_classes import ConllEntry, Sentence

In [ ]:
import pickle

In [ ]:
with open('/content/parser_stat/treebank_test_sets/treebank_data.pickle', 'rb') as f:
    treebanks = pickle.load(f)

In [ ]:
for t, sent_list in treebanks.items():
    print(t, len(sent_list))
    # checking the index uniqueness
    assert len({s.sent_id for s in sent_list}) == len(sent_list)

gsd 601
pud 1000
syntagrus 8800
poetry 728
taiga 881


In [ ]:
treebank_names = ['taiga', 'poetry', 'gsd', 'pud', 'syntagrus']

# Parsing result getting

In [ ]:
import pandas as pd

In [ ]:
parser_names = ["natasha", "udpipe", "spacy", "deeppavlov", "stanza"]

In [ ]:
parser_res = {}
for p in parser_names:
    with open(f'/content/parser_stat/pickle_results/{p}_t.pickle', 'rb') as f:
       parser_res[p] = pickle.load(f)

In [ ]:
for p in parser_names:
  for t in treebanks:
    assert len(treebanks[t]) == len(parser_res[p][t])

In [ ]:
[(t.id, t.parent_id, t.relation) for t in parser_res['stanza']['syntagrus'][1].tokens]

[('1', '11', 'acl'),
 ('2', '3', 'case'),
 ('3', '1', 'obl'),
 ('4', '6', 'case'),
 ('5', '6', 'amod'),
 ('6', '1', 'obl'),
 ('7', '6', 'nmod'),
 ('8', '11', 'amod'),
 ('9', '10', 'cc'),
 ('10', '8', 'conj'),
 ('11', '12', 'nsubj'),
 ('12', '0', 'root'),
 ('13', '12', 'obl'),
 ('14', '15', 'cc'),
 ('15', '13', 'conj'),
 ('16', '15', 'nmod'),
 ('17', '12', 'punct')]

In [ ]:
[i for (i, t) in enumerate(parser_res['stanza']['syntagrus'][95].tokens) if t.id == '1']

[0]

In [ ]:
parser_res['udpipe']['poetry'][1].sent_id

'xix__plesheev__plesh-029-2'

In [ ]:
parser_res['stanza']['syntagrus'][1].sent_id

'2003Armeniya.xml_2'

In [ ]:
for t in treebank_names:
  for p in parser_names:
      print(t, p, len([i for (i, sent) in enumerate(parser_res[p][t])
       if len([t.relation for t in sent.tokens if t.relation == "root"]) > 1]))

taiga natasha 68
taiga udpipe 0
taiga spacy 65
taiga deeppavlov 68
taiga stanza 0
poetry natasha 140
poetry udpipe 0
poetry spacy 21
poetry deeppavlov 50
poetry stanza 0
gsd natasha 70
gsd udpipe 0
gsd spacy 7
gsd deeppavlov 19
gsd stanza 0
pud natasha 98
pud udpipe 0
pud spacy 5
pud deeppavlov 3
pud stanza 0
syntagrus natasha 1078
syntagrus udpipe 0
syntagrus spacy 259
syntagrus deeppavlov 127
syntagrus stanza 0


In [ ]:
[i for (i, sent) in enumerate(parser_res['stanza']['syntagrus'])
    if len([t.relation for t in sent.tokens if t.relation == "root"]) > 2]

[]

In [ ]:
parser_res['stanza']['syntagrus'][6424].sent_id

'2013Martovskaya_revolyutsiya.xml_88'

In [ ]:
[(t.id, t.parent_id, t.relation) for t in parser_res['stanza']['syntagrus'][1].tokens]

[('1', '11', 'acl'),
 ('2', '3', 'case'),
 ('3', '1', 'obl'),
 ('4', '6', 'case'),
 ('5', '6', 'amod'),
 ('6', '1', 'obl'),
 ('7', '6', 'nmod'),
 ('8', '11', 'amod'),
 ('9', '10', 'cc'),
 ('10', '8', 'conj'),
 ('11', '12', 'nsubj'),
 ('12', '0', 'root'),
 ('13', '12', 'obl'),
 ('14', '15', 'cc'),
 ('15', '13', 'conj'),
 ('16', '15', 'nmod'),
 ('17', '12', 'punct')]

In [ ]:
def shift_token_id(sentence):
  first_token_shift = 0
  for i, t in enumerate(sentence.tokens):
    if t.id == '1':
      first_token_shift = i
    shift_id = str(int(t.id) + first_token_shift)
    if t.parent_id != '0':
      shift_parent_id = str(int(t.parent_id) + first_token_shift)
    else:
      shift_parent_id = '0'
    #print(shift_id, t.id, "    ", shift_parent_id, t.parent_id)
    t.id = shift_id
    t.parent_id = shift_parent_id

In [ ]:
for t in treebank_names:
  for p in parser_names:
    for s in parser_res[p][t]:
      shift_token_id(s)

In [ ]:
[(t.id, t.parent_id, t.relation) for t in parser_res['stanza']['syntagrus'][6424].tokens]

[('1', '0', 'root'),
 ('2', '3', 'advmod'),
 ('3', '1', 'nsubj'),
 ('4', '3', 'obl'),
 ('5', '9', 'punct'),
 ('6', '9', 'cc'),
 ('7', '8', 'amod'),
 ('8', '9', 'nsubj'),
 ('9', '1', 'conj'),
 ('10', '12', 'case'),
 ('11', '12', 'det'),
 ('12', '9', 'obl'),
 ('13', '17', 'punct'),
 ('14', '17', 'amod'),
 ('15', '17', 'amod'),
 ('16', '17', 'amod'),
 ('17', '12', 'parataxis'),
 ('18', '19', 'punct'),
 ('19', '17', 'parataxis'),
 ('20', '19', 'flat'),
 ('21', '22', 'amod'),
 ('22', '19', 'nmod'),
 ('23', '19', 'punct'),
 ('24', '27', 'punct'),
 ('25', '27', 'amod'),
 ('26', '27', 'amod'),
 ('27', '17', 'conj'),
 ('28', '29', 'punct'),
 ('29', '27', 'parataxis'),
 ('30', '29', 'flat'),
 ('31', '29', 'punct'),
 ('32', '35', 'punct'),
 ('33', '35', 'amod'),
 ('34', '35', 'amod'),
 ('35', '17', 'conj'),
 ('36', '37', 'punct'),
 ('37', '35', 'parataxis'),
 ('38', '37', 'flat'),
 ('39', '37', 'punct'),
 ('40', '41', 'punct'),
 ('41', '17', 'conj'),
 ('42', '41', 'nmod'),
 ('43', '44', 'punct'),

## Проверка на соответствие количества предложений

In [ ]:
for p in parser_res:
    for t in treebanks:
        if t in parser_res[p]:
            assert len(treebanks[t]) == len(parser_res[p][t]), f"Несовпадение для {p}/{t}"
print("Все проверки пройдены.")

Все проверки пройдены.


# Creating token start-end pairs

be_edges - set of dependency tree edges in begin-end format

In [ ]:
def create_sent_be_nodes(sent, text_transform):
    token_begin_end = []
    sent_text = text_transform(sent.text)
    original_sent_text = text_transform(sent.text)
    del_prefix_len = 0
    tokens = [t for t in sent.tokens if '.' not in t.id]
    for t_i, t in enumerate(tokens):
        token_text = text_transform(t.form)
        t_start = sent_text.find(token_text)
        if t_start == -1:
            print("Error:", sent.sent_id, f"sent_text:{sent_text}, t:{token_text}", t_i)
        else:
            b, e = (del_prefix_len + t_start,
                                  del_prefix_len + t_start + len(token_text))
            token_begin_end.append((t, (b, e)))
            del_prefix_len += t_start + len(token_text)
            sent_text = sent_text[t_start + len(token_text):]
            assert text_transform(original_sent_text[b:e]) == text_transform(tokens[t_i].form)
    sent_text = text_transform(sent.text)
    return token_begin_end

In [ ]:
def create_sent_be_edges(sent_be_tokens):
    sent_be_res = {}
    for t_id, (t, t_be) in enumerate(sent_be_tokens): # ellipsis are deleted, so index in sent_be_tokens = token_id
      parent_id = t.parent_id
      if parent_id == '0': # root
        parent_be = (-1, -1)
      else:
        _, parent_be = sent_be_tokens[int(parent_id) - 1]
      sent_be_res[t_be] = (parent_be, t.relation)
    return sent_be_res

In [ ]:
from collections import OrderedDict

In [ ]:
be_treebanks = {}
for treebank_n in treebank_names:
    be_treebanks[treebank_n] = OrderedDict()
    for i, sent in enumerate(treebanks[treebank_n]):
        be_sent = create_sent_be_nodes(sent, lambda text: text.lower())
        be_treebanks[treebank_n][sent.sent_id] = create_sent_be_edges(be_sent)

In [ ]:
be_treebanks['syntagrus']['2003Artist_mimansa.xml_130']

{(0, 2): ((16, 20), 'nsubj'),
 (3, 15): ((16, 20), 'advmod'),
 (16, 20): ((-1, -1), 'root'),
 (21, 24): ((25, 34), 'det'),
 (25, 34): ((16, 20), 'obj'),
 (35, 43): ((16, 20), 'advmod'),
 (43, 44): ((16, 20), 'punct')}

{(0, 2): ((16, 20), 'nsubj'),

 (3, 15): ((16, 20), 'advmod'),

 (16, 20): ((-1, -1), 'root'),

 (21, 24): ((25, 34), 'det'),

 (25, 34): ((16, 20), 'obj'),

 (35, 43): ((16, 20), 'advmod'),

 (43, 44): ((16, 20), 'punct')}


In [ ]:
be_parser_res = {p: {} for p in parser_names}
for p in parser_names:
  if p == 'deeppavlov':
    transform_fun = lambda text: text.lower().replace('``', '"').replace("''", '"')
  else:
    transform_fun = lambda text: text.lower().replace("''", '"')
  for t in treebank_names:
    be_parser_res[p][t] = OrderedDict()
    for i, sent in enumerate(parser_res[p][t]):
        be_sent = create_sent_be_nodes(sent, transform_fun)
        be_parser_res[p][t][sent.sent_id] = create_sent_be_edges(be_sent)

In [ ]:
be_parser_res['stanza']['syntagrus']['2003Armeniya.xml_2']

{(0, 11): ((87, 96), 'acl'),
 (12, 13): ((14, 24), 'case'),
 (14, 24): ((0, 11), 'obl'),
 (25, 27): ((39, 46), 'case'),
 (28, 38): ((39, 46), 'amod'),
 (39, 46): ((0, 11), 'obl'),
 (47, 54): ((39, 46), 'nmod'),
 (55, 67): ((87, 96), 'amod'),
 (68, 69): ((70, 86), 'cc'),
 (70, 86): ((55, 67), 'conj'),
 (87, 96): ((97, 105), 'nsubj'),
 (97, 105): ((-1, -1), 'root'),
 (106, 112): ((97, 105), 'obl'),
 (113, 114): ((115, 120), 'cc'),
 (115, 120): ((106, 112), 'conj'),
 (121, 127): ((115, 120), 'nmod'),
 (127, 128): ((97, 105), 'punct')}

((0, 2), ((16, 20), 'nsubj')),

((3, 9), ((16, 20), 'obl')),

((9, 10), ((10, 15), 'punct')),

((10, 15), ((3, 9), 'conj')),

((16, 20), ((-1, -1), 'root')),

((21, 24), ((25, 34), 'det')),

((25, 34), ((16, 20), 'obj')),

((35, 43), ((16, 20), 'advmod')),

((43, 44), ((16, 20), 'punct'))]

In [ ]:
for p in be_parser_res:
    for t in be_treebanks:
        if t in be_parser_res[p]:
            assert len(be_treebanks[t]) == len(be_parser_res[p][t]), f"Несовпадение для {p}/{t}"
print("Все проверки пройдены.")

Все проверки пройдены.


## Сохранение

In [ ]:
import pickle

with open('be_treebanks.pkl', 'wb') as f:
    pickle.dump(be_treebanks, f)

In [ ]:
with open('/content/be_parser_res.pkl', 'wb') as f:
    pickle.dump(be_parser_res, f)

# Векторизация графов

Graph2Vec + clustering для эталонных dependency-графов

Задача: выделение кластеров среди gold/эталонных разборов предложений

Рекомендуемый запуск: Google Colab, отдельное conda-окружение

Python 3.7.

Важное замечание об окружении
`karateclub==1.3.3` опубликован как пакет для Python 3.7 и зависит от старого стека
`networkx/gensim/scipy/sklearn`. В современном Colab Python 3.10/3.11 установка часто ломается из-за несовместимых версий NumPy/SciPy/Gensim.

Поэтому ниже используется отдельное conda-окружение через `condacolab`.
После первой ячейки Colab перезапустит runtime.

In [ ]:
try:
    import condacolab  # type: ignore
    print("condacolab уже установлен")
except Exception:
    !pip -q install condacolab
    import condacolab  # type: ignore
    condacolab.install()

condacolab уже установлен


Важно: после выполнения этой ячейки используйте python из /usr/local/envs/g2v37/bin/python.

Он не станет ядром ноутбука автоматически, поэтому основной pipeline ниже запускается как отдельный скрипт через conda run.


In [ ]:
!conda create -y -n g2v37 python=3.7 pip
!conda run -n g2v37 python -m pip install --upgrade "pip<24" "setuptools<60" "wheel<0.40"

!conda run -n g2v37 python -m pip install \
    "numpy==1.21.6" \
    "scipy==1.7.3" \
    "scikit-learn==1.0.2" \
    "networkx==2.6.3" \
    "gensim==4.2.0" \
    "pandas==1.3.5" \
    "matplotlib==3.5.3" \
    "karateclub==1.3.3" \
    "umap-learn==0.5.3" \
    "tqdm==4.66.1" \
    "joblib==1.2.0" \
    "pyyaml==6.0"

/bin/bash: line 1: conda: command not found
/bin/bash: line 1: conda: command not found
/bin/bash: line 1: conda: command not found


In [ ]:
# Проверка окружения.
%%writefile check_env.py
import sys, platform
import numpy, scipy, sklearn, networkx, gensim, karateclub

print("Python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
print("networkx:", networkx.__version__)
print("gensim:", gensim.__version__)
print("karateclub:", getattr(karateclub, "__version__", "unknown"))

Writing check_env.py


In [ ]:
!conda run -n g2v37 python check_env.py

Python: 3.7.12 | packaged by conda-forge | (default, Oct 26 2021, 06:08:21) 
[GCC 9.4.0]
Platform: Linux-6.6.122+-x86_64-with-debian-bookworm-sid
numpy: 1.21.6
scipy: 1.7.3
sklearn: 1.0.2
networkx: 2.6.3
gensim: 4.2.0
karateclub: 1.3.3



### 1. Подготовка входных данных

Ожидаемый вход: файл `be_treebanks.pkl`, полученный из предыдущей части ноутбука.

Он должен содержать структуру вида:

```python
be_treebanks = {
     "gsd": {
         sent_id: deps_dict,
         ...
     },
     "syntagrus": {...},
     ...
 }
 ```

 Если есть исходные gold-предложения с токенами и признаками UPOS/morph, лучше использовать вариант
 `build_gold_graphs_from_sentences`.

 Если доступны только span-зависимости, используется fallback,
 где node feature строится не из span, а из роли токена в дереве: incoming dependency relation + degree.
 Это лучше, чем `(begin, end)`.

## Программная утилита

In [ ]:
%%writefile dep_metrics.py

import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
import signal
from contextlib import contextmanager
from collections import defaultdict, deque

# ----------------------------------------------------------------------
# Вспомогательная функция для поиска корня в словаре зависимостей
# ----------------------------------------------------------------------
def find_root(deps):
    """Ищет корень в словаре зависимостей {token: (head, rel)}."""
    for token, (head, rel) in deps.items():
        if head == (-1, -1) or rel == 'root':
            return token
    # fallback: первый токен, если корень не найден
    return next(iter(deps)) if deps else None

# ----------------------------------------------------------------------
# Построение ориентированного графа из словаря зависимостей
# ----------------------------------------------------------------------
def networkx_formatter(dict_Graph, nx_Graph, show=False):
    """
    Преобразует словарь {node: (parent, label)} в networkx.DiGraph.
    Узлы - кортежи (start, end). Если parent = (-1,-1), то это корень.
    Возвращает nx_Graph.
    """
    nx_Graph.clear()
    for node, value in dict_Graph.items():
        if isinstance(value, tuple) and len(value) == 2:
            head, label = value
            nx_Graph.add_node(node)
            if head != (-1, -1):
                nx_Graph.add_edge(head, node, label=label)
        else:
            print(f"Пропущено значение для узла {value} (некорректная структура)\n")
    if show:
        # Вспомогательная отрисовка (упрощённая)
        plt.figure(figsize=(8, 6))
        try:
            pos = nx.planar_layout(nx_Graph, scale=2)
        except:
            pos = nx.spring_layout(nx_Graph)
        nx.draw(nx_Graph, pos, with_labels=True, node_size=700, node_color="lightblue", font_size=6, font_weight="bold")
    return nx_Graph

# ----------------------------------------------------------------------
# WL-ядро для вычисления сходства двух графов (списки графов)
# ----------------------------------------------------------------------
def new_wl_ker1(results, references, h=3):
    """
    Вычисляет косинусное сходство между гистограммами WL-меток
    для каждой пары графов из списков results и references.
    Возвращает numpy-массив similarities длиной len(results).
    """
    assert len(results) == len(references), "Размеры списков должны совпадать"

    feature_maps_results = [defaultdict(int) for _ in results]
    feature_maps_refs = [defaultdict(int) for _ in references]

    # Инициализация: метки степеней вершин
    for i, G in enumerate(results):
        for node in G.nodes():
            label = str(G.degree(node))
            feature_maps_results[i][label] += 1
    for i, G in enumerate(references):
        for node in G.nodes():
            label = str(G.degree(node))
            feature_maps_refs[i][label] += 1

    # Сохранение предыдущих меток
    prev_labels_results = [{} for _ in results]
    prev_labels_refs = [{} for _ in references]
    for i, G in enumerate(results):
        prev_labels_results[i] = {node: str(G.degree(node)) for node in G.nodes()}
    for i, G in enumerate(references):
        prev_labels_refs[i] = {node: str(G.degree(node)) for node in G.nodes()}

    for _ in range(h):
        # Обновление меток в results
        new_labels_results = [{} for _ in results]
        for i, G in enumerate(results):
            for node in G.nodes():
                neighbor_info = []
                for neighbor in G.neighbors(node):
                    neighbor_label = prev_labels_results[i][neighbor]
                    edge_label = G[node][neighbor].get('label', '')
                    neighbor_info.append(f"{neighbor_label}_{edge_label}")
                neighbor_info_sorted = sorted(neighbor_info)
                if not neighbor_info_sorted:
                    neighbor_info_sorted = ['none']
                new_label = f"{prev_labels_results[i][node]}_{'_'.join(neighbor_info_sorted)}"
                new_labels_results[i][node] = new_label
                feature_maps_results[i][new_label] += 1
        prev_labels_results = new_labels_results

        # Обновление меток в references
        new_labels_refs = [{} for _ in references]
        for i, G in enumerate(references):
            for node in G.nodes():
                neighbor_info = []
                for neighbor in G.neighbors(node):
                    neighbor_label = prev_labels_refs[i][neighbor]
                    edge_label = G[node][neighbor].get('label', '')
                    neighbor_info.append(f"{neighbor_label}_{edge_label}")
                neighbor_info_sorted = sorted(neighbor_info)
                if not neighbor_info_sorted:
                    neighbor_info_sorted = ['none']
                new_label = f"{prev_labels_refs[i][node]}_{'_'.join(neighbor_info_sorted)}"
                new_labels_refs[i][node] = new_label
                feature_maps_refs[i][new_label] += 1
        prev_labels_refs = new_labels_refs

    # Нормализация гистограмм
    for i in range(len(results)):
        total = sum(feature_maps_results[i].values())
        if total > 0:
            for key in feature_maps_results[i]:
                feature_maps_results[i][key] /= total
    for i in range(len(references)):
        total = sum(feature_maps_refs[i].values())
        if total > 0:
            for key in feature_maps_refs[i]:
                feature_maps_refs[i][key] /= total

    # Косинусное сходство
    similarities = np.zeros(len(results))
    for i in range(len(results)):
        dot_product = sum(feature_maps_results[i].get(k, 0) * feature_maps_refs[i].get(k, 0)
                         for k in set(feature_maps_results[i]) | set(feature_maps_refs[i]))
        norm_i = np.sqrt(sum(v ** 2 for v in feature_maps_results[i].values()))
        norm_j = np.sqrt(sum(v ** 2 for v in feature_maps_refs[i].values()))
        if norm_i * norm_j > 0:
            similarities[i] = dot_product / (norm_i * norm_j)
        else:
            similarities[i] = 0.0
    return similarities

# ----------------------------------------------------------------------
# Расстояние редактирования графов с таймаутом
# ----------------------------------------------------------------------
class TimeoutException(Exception):
    pass

@contextmanager
def time_limit(seconds):
    def signal_handler(signum, frame):
        raise TimeoutException("Timed out!")
    signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)

def graph_edit_distance_with_labels(G1, G2, timeout=30, use_approx=True):
    """
    Вычисляет GED с учётом меток узлов ('word') и рёбер ('label').
    При превышении таймаута возвращает приближённое значение.
    """
    def node_match(n1, n2):
        return n1.get('word', '') == n2.get('word', '')
    def edge_match(e1, e2):
        return e1.get('label', '') == e2.get('label', '')
    try:
        with time_limit(timeout):
            return nx.graph_edit_distance(G1, G2, node_match=node_match, edge_match=edge_match)
    except TimeoutException:
        print(f"  Превышен таймаут {timeout}с для графов размером {G1.number_of_nodes()} и {G2.number_of_nodes()}")
        if use_approx:
            approx = abs(G1.number_of_nodes() - G2.number_of_nodes()) + \
                     abs(G1.number_of_edges() - G2.number_of_edges())
            print(f"  Возвращаем приближённое значение: {approx}")
            return approx
        else:
            return None
    except Exception as e:
        print(f"  Ошибка при вычислении GED: {e}")
        return None

# ----------------------------------------------------------------------
# Простая структурная метрика искажения
# ----------------------------------------------------------------------
def tree_depth_and_branching(deps, root):
    """Вычисляет глубину дерева и средний коэффициент ветвления."""
    if not deps or root is None:
        return 0, 0.0
    G = nx.DiGraph()
    G.add_nodes_from(deps.keys())
    for token, (head, _) in deps.items():
        if head != (-1, -1):
            G.add_edge(head, token)
    if root not in G:
        return 0, 0.0
    depths = {}
    queue = [(root, 0)]
    visited = {root}
    while queue:
        node, d = queue.pop(0)
        depths[node] = d
        for child in G.successors(node):
            if child not in visited:
                visited.add(child)
                queue.append((child, d + 1))
    depth = max(depths.values()) if depths else 0
    children_count = [G.out_degree(n) for n in G.nodes]
    avg_branch = np.mean(children_count) if children_count else 0.0
    return depth, avg_branch

def structural_distortion_score(gold_deps, parser_deps, gold_root, parser_root):
    """
    Оценка структурного искажения в диапазоне [0, 1].
    Учитывает:
      - долю узлов с изменённым родителем,
      - долю узлов с изменением глубины > 1,
      - относительное изменение общей глубины,
      - относительное изменение среднего ветвления.
    """
    gold_tokens = set(gold_deps.keys())
    parser_tokens = set(parser_deps.keys())
    common_tokens = gold_tokens & parser_tokens
    if not common_tokens:
        return 1.0

    # Доля узлов с изменённым родителем
    parent_changed = 0
    for t in common_tokens:
        if gold_deps[t][0] != parser_deps[t][0]:
            parent_changed += 1
    parent_ratio = parent_changed / len(common_tokens)

    # Изменение глубины
    def compute_depths(deps, root):
        G = nx.DiGraph()
        G.add_nodes_from(deps.keys())
        for t, (h, _) in deps.items():
            if h != (-1, -1):
                G.add_edge(h, t)
        depths = {}
        if root in G:
            queue = [(root, 0)]
            visited = {root}
            while queue:
                node, d = queue.pop(0)
                depths[node] = d
                for child in G.successors(node):
                    if child not in visited:
                        visited.add(child)
                        queue.append((child, d + 1))
        return depths
    gold_depths = compute_depths(gold_deps, gold_root)
    parser_depths = compute_depths(parser_deps, parser_root)

    depth_changed_gt1 = 0
    for t in common_tokens:
        gd = gold_depths.get(t, 0)
        pd = parser_depths.get(t, 0)
        if abs(gd - pd) > 1:
            depth_changed_gt1 += 1
    depth_ratio = depth_changed_gt1 / len(common_tokens)

    # Общая глубина и ветвление
    gold_depth, gold_branch = tree_depth_and_branching(gold_deps, gold_root)
    parser_depth, parser_branch = tree_depth_and_branching(parser_deps, parser_root)
    depth_diff = abs(gold_depth - parser_depth) / max(gold_depth, parser_depth, 1)
    branch_diff = abs(gold_branch - parser_branch) / max(gold_branch, parser_branch, 1e-6)

    # Взвешенная сумма
    score = (0.3 * parent_ratio +
             0.3 * depth_ratio +
             0.2 * depth_diff +
             0.2 * branch_diff)
    return min(score, 1.0)

# ----------------------------------------------------------------------
# Иерархическая визуализация пары деревьев (эталон и парсер)
# ----------------------------------------------------------------------
def draw_hierarchical_graph(ax, graph_dict, sentence_text, title, forced_root=None):
    """
    Рисует на оси ax граф зависимостей.
    graph_dict: словарь {token_span: (parent_span, relation)}.
    sentence_text: строка предложения.
    title: заголовок.
    forced_root: принудительно заданный корень для упорядочивания (опционально).
    """
    # --- построение графа ---
    G = nx.Graph()
    coord_to_word = {}
    children = defaultdict(list)
    parents = {}
    explicit_roots = set()

    for node, value in graph_dict.items():
        start, end = node
        if end <= len(sentence_text):
            word = sentence_text[start:end].strip()
            if not word:
                word = sentence_text[start:end]
            coord_to_word[node] = word
        else:
            coord_to_word[node] = f"({start},{end})"

        if isinstance(value, tuple) and len(value) == 2:
            head, label = value
            G.add_node(node, word=coord_to_word[node])
            if head == (-1, -1):
                explicit_roots.add(node)
            else:
                G.add_edge(head, node, label=label, parent=head, child=node)
                children[head].append(node)
                parents[node] = head

    if G.number_of_nodes() == 0:
        ax.text(0.5, 0.5, "Пустой граф", ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        ax.axis('off')
        return

    # --- компоненты связности ---
    components = list(nx.connected_components(G))
    component_roots = []
    node_to_level = {}
    node_to_component_idx = {}

    for comp_idx, comp_nodes in enumerate(components):
        # выбор корня компоненты
        root_candidates = [n for n in comp_nodes if n in explicit_roots]
        if not root_candidates:
            no_parent = [n for n in comp_nodes if n not in parents]
            if no_parent:
                root_candidates = no_parent
            else:
                root_candidates = [next(iter(comp_nodes))]
        if forced_root is not None and forced_root in comp_nodes:
            root = forced_root
        else:
            root = root_candidates[0]
        component_roots.append(root)

        # BFS для определения уровней
        queue = deque([(root, 0)])
        visited = {root}
        level_map = {root: 0}
        while queue:
            node, lvl = queue.popleft()
            for neighbor in G.neighbors(node):
                if neighbor not in visited:
                    visited.add(neighbor)
                    level_map[neighbor] = lvl + 1
                    queue.append((neighbor, lvl + 1))
        for node, lvl in level_map.items():
            node_to_level[node] = lvl
            node_to_component_idx[node] = comp_idx

    # --- размещение узлов ---
    comp_level_counts = []
    for comp_idx, root in enumerate(component_roots):
        nodes_in_comp = [n for n, cidx in node_to_component_idx.items() if cidx == comp_idx]
        levels = [node_to_level[n] for n in nodes_in_comp]
        max_level = max(levels) if levels else 0
        level_count = [0] * (max_level + 1)
        for node in nodes_in_comp:
            lvl = node_to_level[node]
            level_count[lvl] += 1
        comp_level_counts.append(level_count)

    pos = {}
    inter_component_gap = 8.0
    current_x = 0.0

    for comp_idx, level_count in enumerate(comp_level_counts):
        nodes_in_comp = [n for n, cidx in node_to_component_idx.items() if cidx == comp_idx]
        if not nodes_in_comp:
            continue
        max_width = max(level_count) if level_count else 1
        x_spacing = 2.5
        level_to_nodes = defaultdict(list)
        for node in nodes_in_comp:
            lvl = node_to_level[node]
            level_to_nodes[lvl].append(node)
        max_lvl = max(level_to_nodes.keys())
        for lvl, nodes in level_to_nodes.items():
            num = len(nodes)
            center_x = current_x + (max_width * x_spacing) / 2
            if num == 1:
                x_positions = [center_x]
            else:
                start_x = center_x - (num - 1) * x_spacing / 2
                x_positions = [start_x + i * x_spacing for i in range(num)]
            for node, x in zip(nodes, x_positions):
                y = (max_lvl - lvl) * 2.5
                pos[node] = (x, y)
        current_x += max_width * x_spacing + inter_component_gap

    # --- отрисовка рёбер ---
    for u, v, data in G.edges(data=True):
        if data.get('parent') == u and data.get('child') == v:
            start_node, end_node = u, v
        elif data.get('parent') == v and data.get('child') == u:
            start_node, end_node = v, u
        else:
            continue
        if start_node not in pos or end_node not in pos:
            continue
        start_pos = pos[start_node]
        end_pos = pos[end_node]
        label = data.get('label', '')
        ax.annotate('', xy=end_pos, xytext=start_pos,
                    arrowprops=dict(arrowstyle='->', color='black', lw=2.5,
                                    shrinkA=25, shrinkB=25))
        if label:
            mid_x = (start_pos[0] + end_pos[0]) / 2
            mid_y = (start_pos[1] + end_pos[1]) / 2
            ax.text(mid_x, mid_y, label,
                    fontsize=10, ha='center', va='center',
                    bbox=dict(boxstyle="round,pad=0.3", facecolor='white',
                             edgecolor='gray', alpha=0.95),
                    zorder=5)

    # --- отрисовка узлов ---
    for node, (x, y) in pos.items():
        word = coord_to_word[node]
        text_width = max(1.8, len(word) * 0.24 + 0.8)
        text_height = 0.8
        rect = patches.FancyBboxPatch(
            (x - text_width/2, y - text_height/2),
            text_width, text_height,
            boxstyle="round,pad=0.1",
            facecolor='lightblue',
            edgecolor='black',
            linewidth=2.5,
            zorder=3
        )
        ax.add_patch(rect)
        ax.text(x, y, word,
                ha='center', va='center',
                fontsize=20, fontweight='bold', color='black', zorder=4)

    ax.set_title(title)
    ax.set_aspect('equal')
    ax.axis('off')
    if pos:
        x_coords = [p[0] for p in pos.values()]
        y_coords = [p[1] for p in pos.values()]
        margin = 2.0
        ax.set_xlim(min(x_coords) - margin, max(x_coords) + margin)
        ax.set_ylim(min(y_coords) - 1.2, max(y_coords) + 1.2)

Writing dep_metrics.py


In [ ]:
%%writefile run_utility.py
"""
Утилита анализа структурных ошибок синтаксических деревьев.
Режимы:
  filter – отбор предложений по порогам метрик, визуализация пар gold/parser.
  cluster – векторизация графов (Graph2Vec), плотностная кластеризация (DBSCAN),
            сохранение изображений деревьев по кластерам и отчётов.

Конфигурация задаётся через YAML-файл.
"""

import argparse
import os
import pickle
import sys
import time
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import yaml

from dep_metrics import (
    draw_hierarchical_graph,
    find_root,
    graph_edit_distance_with_labels,
    networkx_formatter,
    new_wl_ker1,
    structural_distortion_score,
)
from graph2vec_clustering import (
    ExperimentConfig,
    fit_graph2vec,
    flatten_be_treebanks,
    run_clustering_suite,
    save_outputs,
    set_seed,
)

from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
)
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.neighbors import NearestNeighbors

try:
    from hdbscan import HDBSCAN
    HDBSCAN_AVAILABLE = True
except ImportError:
    try:
        from sklearn.cluster import HDBSCAN  # sklearn >= 1.1.0
        HDBSCAN_AVAILABLE = True
    except ImportError:
        HDBSCAN_AVAILABLE = False

# ---------------------------------------------------------------------------
# Вспомогательные функции
# ---------------------------------------------------------------------------

def load_pickle(path: str) -> Any:
    with open(path, "rb") as f:
        return pickle.load(f)


def load_config(config_path: str) -> Dict[str, Any]:
    with open(config_path, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    # обязательные поля
    if "input" not in cfg or "gold" not in cfg["input"]:
        raise ValueError("Config must specify input.gold")
    if "output_dir" not in cfg:
        raise ValueError("Config must specify output_dir")
    if "mode" not in cfg:
        raise ValueError("Config must specify mode ('filter' or 'cluster')")
    return cfg


def flatten_treebank_dict(treebank_dict: Dict[str, OrderedDict]) -> Dict[str, Any]:
    """Превращает {treebank -> OrderedDict(sent_id -> deps)} в {sent_id -> deps}."""
    flat = {}
    for tb_data in treebank_dict.values():
        flat.update(tb_data)
    return flat


def load_datasets(cfg: Dict[str, Any]):
    """
    Возвращает gold_raw (словарь по корпусам) и,
    если задан parser, то parser_raw для указанного парсера.
    """
    gold_raw = load_pickle(cfg["input"]["gold"])
    parser_raw = None
    parser_section = cfg["input"].get("parser")
    if parser_section:
        all_parsers = load_pickle(parser_section)
        name = cfg["input"].get("parser_name")
        if name:
            parser_raw = all_parsers[name]
        else:
            parser_raw = all_parsers
    return gold_raw, parser_raw

def compute_internal_metrics(X, labels, noise_label=-1):
    """
    Вычисляет коэффициент силуэта, индекс Davies–Bouldin и Calinski–Harabasz
    по не‑шумовым точкам.
    """
    mask = labels != noise_label
    if mask.sum() <= 1 or len(set(labels[mask])) < 2:
        return {"silhouette": None, "davies_bouldin": None, "calinski_harabasz": None}
    X_sub = X[mask]
    labels_sub = labels[mask]
    return {
        "silhouette": silhouette_score(X_sub, labels_sub),
        "davies_bouldin": davies_bouldin_score(X_sub, labels_sub),
        "calinski_harabasz": calinski_harabasz_score(X_sub, labels_sub),
    }

# ---------------------------------------------------------------------------
# Режим фильтрации
# ---------------------------------------------------------------------------

def filter_pipeline(cfg: Dict[str, Any]) -> None:
    gold_raw, parser_raw = load_datasets(cfg)
    if parser_raw is None:
        print("Error: filter mode requires parser data.")
        return

    gold_flat = flatten_treebank_dict(gold_raw)
    parser_flat = flatten_treebank_dict(parser_raw)
    common_ids = sorted(set(gold_flat.keys()) & set(parser_flat.keys()))
    print(f"Loaded {len(common_ids)} common sentences.")

    # Загрузка опционального маппинга текстов
    text_mapping = None
    if "text_mapping" in cfg["input"]:
        text_mapping = load_pickle(cfg["input"]["text_mapping"])

    # Предзагрузка Graph2Vec-эмбеддингов при необходимости
    g2v_gold_emb = g2v_parser_emb = None
    if cfg.get("metrics", {}).get("graph2vec", {}).get("enabled", False):
        g2v_cfg = cfg["metrics"]["graph2vec"]
        g2v_gold_emb = load_pickle(g2v_cfg["gold_embeddings"])
        g2v_parser_emb = load_pickle(g2v_cfg["parser_embeddings"])

    output_dir = Path(cfg["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    records = []
    for sid in common_ids:
        gold_deps = gold_flat[sid]
        parser_deps = parser_flat[sid]
        G_gold = networkx_formatter(gold_deps, nx.DiGraph())
        G_parser = networkx_formatter(parser_deps, nx.DiGraph())

        wl_val = ged_val = struct_val = g2v_dist = None

        if cfg["metrics"].get("wl", {}).get("enabled", False):
            wl_val = new_wl_ker1(
                [G_gold], [G_parser],
                h=cfg["metrics"]["wl"].get("iterations", 3),
            )[0]

        if cfg["metrics"].get("ged", {}).get("enabled", False):
            ged_val = graph_edit_distance_with_labels(
                G_gold, G_parser,
                timeout=cfg["metrics"]["ged"].get("timeout", 60),
            )

        if cfg["metrics"].get("structural", {}).get("enabled", False):
            gold_root = find_root(gold_deps)
            parser_root = find_root(parser_deps)
            struct_val = structural_distortion_score(
                gold_deps, parser_deps, gold_root, parser_root
            )

        if g2v_gold_emb is not None and g2v_parser_emb is not None:
            v_gold = g2v_gold_emb.get(sid)
            v_parser = g2v_parser_emb.get(sid)
            if v_gold is not None and v_parser is not None:
                cos_sim = np.dot(v_gold, v_parser) / (
                    np.linalg.norm(v_gold) * np.linalg.norm(v_parser) + 1e-12
                )
                g2v_dist = 1.0 - cos_sim

        records.append(
            {
                "sent_id": sid,
                "wl": wl_val,
                "ged": ged_val,
                "structural": struct_val,
                "graph2vec_dist": g2v_dist,
            }
        )

    df = pd.DataFrame(records)

    # Применение условий фильтрации
    filter_cfg = cfg.get("filter", {})
    combine = filter_cfg.get("combine", "OR").upper()
    conditions = filter_cfg.get("conditions", [])

    def check(row):
        flags = []
        for cond in conditions:
            m = cond["metric"]
            lo = cond.get("min", -float("inf"))
            hi = cond.get("max", float("inf"))
            val = row.get(m)
            if val is None:
                flags.append(False)
            else:
                flags.append(lo <= val <= hi)
        return all(flags) if combine == "AND" else any(flags)

    df["pass"] = df.apply(check, axis=1)
    passed = df[df["pass"]]
    print(f"Filtering: {len(passed)} sentences passed out of {len(df)}")

    # Визуализация
    for _, row in passed.iterrows():
        sid = row["sent_id"]
        gold_deps = gold_flat[sid]
        parser_deps = parser_flat[sid]

        sentence_text = text_mapping.get(sid, "") if text_mapping else ""

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=cfg.get("visualization", {}).get("fig_size", [20, 14]))
        gold_root = find_root(gold_deps)
        draw_hierarchical_graph(ax1, gold_deps, sentence_text, f"Gold: {sid}")
        draw_hierarchical_graph(ax2, parser_deps, sentence_text, f"Parser: {sid}", forced_root=gold_root)

        metric_str = ", ".join(
            f"{m}: {row[m]:.4f}" for m in ["wl", "ged", "structural", "graph2vec_dist"]
            if row[m] is not None
        )
        plt.suptitle(metric_str, fontsize=cfg.get("visualization", {}).get("font_size", 12))
        out_path = output_dir / f"{sid}.png"
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    df.to_csv(output_dir / "filter_metrics.csv", index=False)
    print("Done.")


# ---------------------------------------------------------------------------
# Режим кластеризации
# ---------------------------------------------------------------------------

def cluster_pipeline(cfg: Dict[str, Any]) -> None:

    output_dir = Path(cfg["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    gold_raw, _ = load_datasets(cfg)

    # Собираем словарь предложений для визуализации
    sent_to_deps = {}
    for tb_data in gold_raw.values():
        sent_to_deps.update(tb_data)

    # Настройки векторизации
    vec_cfg = cfg.get("vectorization", {})
    g2v_cfg = ExperimentConfig(
        seed=cfg.get("seed", 42),
        node_feature_mode=vec_cfg.get("node_feature_mode", "dep_degree"),
        dimensions=vec_cfg.get("dimensions", 128),
        wl_iterations=vec_cfg.get("wl_iterations", 3),
        epochs=vec_cfg.get("epochs", 80),
        workers=vec_cfg.get("workers", 2),
        min_count=vec_cfg.get("min_count", 2),
        learning_rate=vec_cfg.get("learning_rate", 0.025),
        down_sampling=vec_cfg.get("down_sampling", 0.0001),
        clustering_k_min=vec_cfg.get("k_min", 2),
        clustering_k_max=vec_cfg.get("k_max", 15),
        dbscan_min_samples=vec_cfg.get("dbscan_min_samples", 10),
    )

    set_seed(g2v_cfg.seed)

    # 1) Построение графов
    graphs, meta = flatten_be_treebanks(gold_raw, g2v_cfg)

    # 2) Получение эмбеддингов Graph2Vec
    embeddings = fit_graph2vec(graphs, g2v_cfg)

    # 3) Кластеризация выбранным алгоритмом
    clust_cfg = cfg.get("clustering", {})
    method = clust_cfg.get("method", "dbscan").lower()
    compute_metrics = clust_cfg.get("compute_metrics", True)

    # Подготовка данных (стандартизация для DBSCAN, для HDBSCAN тоже можно)
    X = StandardScaler().fit_transform(embeddings)

    if method == "dbscan":
        db_cfg = clust_cfg.get("dbscan", {})
        eps = db_cfg.get("eps", None)
        min_samples = db_cfg.get("min_samples", 10)
        metric = db_cfg.get("metric", "euclidean")

        if eps is None:
            # Автоматический подбор eps по расстоянию до k-го соседа (min_samples)
            n_neighbors = min(min_samples, X.shape[0] - 1)
            nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric=metric)
            nbrs.fit(X)
            distances, _ = nbrs.kneighbors(X)
            kth = np.sort(distances[:, -1])
            eps_quantile = db_cfg.get("eps_quantile", 0.75)
            eps = float(np.quantile(kth, eps_quantile))
            print(f"Auto-selected eps = {eps:.4f} (quantile {eps_quantile})")

        model = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)

    elif method == "hdbscan":
        if not HDBSCAN_AVAILABLE:
            raise ImportError("HDBSCAN is not available. Install 'hdbscan' package or use sklearn>=1.1.0")
        hdb_cfg = clust_cfg.get("hdbscan", {})
        min_cluster_size = hdb_cfg.get("min_cluster_size", 10)
        min_samples = hdb_cfg.get("min_samples", None)
        metric = hdb_cfg.get("metric", "euclidean")
        model = HDBSCAN(min_cluster_size=min_cluster_size,
                        min_samples=min_samples,
                        metric=metric)

    else:
        raise ValueError(f"Unknown clustering method: {method}")

    labels = model.fit_predict(X)

    # 4) Метрики качества (опционально)
    if compute_metrics:
        metrics = compute_internal_metrics(X, labels)
        metrics["method"] = method
        metrics["n_clusters"] = len(set(labels)) - (1 if -1 in labels else 0)
        metrics["noise_count"] = int(np.sum(labels == -1))
        metrics_df = pd.DataFrame([metrics])
        metrics_df.to_csv(Path(cfg["output_dir"]) / "clustering_scores.csv", index=False)
    else:
        # Без метрик сохраним только краткую информацию
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_count = int(np.sum(labels == -1))
        info = {"method": method, "n_clusters": n_clusters, "noise_count": noise_count}
        pd.DataFrame([info]).to_csv(Path(cfg["output_dir"]) / "clustering_info.csv", index=False)

    # 5) Сохранение изображений по кластерам
    output_dir = Path(cfg["output_dir"])
    output_dir.mkdir(parents=True, exist_ok=True)

    # Загрузка текстов предложений, если доступен
    text_mapping = None
    if "text_mapping" in cfg.get("input", {}):
        text_mapping = load_pickle(cfg["input"]["text_mapping"])

    meta["cluster"] = labels
    for g_idx, row in meta.iterrows():
        sid = row["sent_id"]
        cluster_id = int(row["cluster"])
        folder = output_dir / f"cluster_{cluster_id}" if cluster_id != -1 else output_dir / "noise"
        folder.mkdir(exist_ok=True)

        deps = sent_to_deps[sid]
        sentence_text = text_mapping.get(sid, "") if text_mapping else ""

        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        draw_hierarchical_graph(ax, deps, sentence_text, f"Gold: {sid}")
        plt.tight_layout()
        img_path = folder / f"{sid}.png"
        plt.savefig(img_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    # 6) Сводка по кластерам
    labeled = meta.copy()
    labeled["cluster"] = labels
    summary = (
        labeled.groupby("cluster")
        .agg(
            n=("sent_id", "count"),
            treebanks=("treebank", lambda x: dict(pd.Series(x).value_counts())),
            n_nodes_mean=("n_nodes", "mean"),
            n_edges_mean=("n_edges", "mean"),
            examples=("sent_id", lambda x: list(x.head(10))),
        )
        .reset_index()
        .sort_values("n", ascending=False)
    )
    summary.to_csv(output_dir / "cluster_summary.csv", index=False)

    print("Cluster mode finished.")


# ---------------------------------------------------------------------------
# Точка входа
# ---------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser(description="Dependency tree analysis utility")
    parser.add_argument("config", help="Path to YAML configuration file")
    args, unknown = parser.parse_known_args()
    if unknown:
        print(f"Warning: ignored unknown arguments: {unknown}")

    cfg = load_config(args.config)

    mode = cfg["mode"]
    if mode == "filter":
        filter_pipeline(cfg)
    elif mode == "cluster":
        cluster_pipeline(cfg)
    else:
        print(f"Unknown mode: {mode}")


if __name__ == "__main__":
    main()

Overwriting run_utility.py


In [ ]:
%%writefile config_filter.yaml
input:
  gold: "/content/be_treebanks.pkl"          # эталонные деревья (словарь {treebank: OrderedDict{sent_id: ...}})
  parser: "/content/be_parser_res.pkl"       # файл с результатами всех парсеров ({parser_name: {treebank: ...}})
  parser_name: "stanza"                      # имя используемого парсера (обязательно, если задан parser)
  # text_mapping: "/content/text_mapping.pkl"  # опционально
output_dir: "/content/filter_output"
mode: "filter"
metrics:
  wl:
    enabled: true
    iterations: 3
  ged:
    enabled: false
    timeout: 60
  structural:
    enabled: true
filter:
  combine: OR
  conditions:
    - metric: wl
      min: 0.0
      max: 0.5
    - metric: structural
      min: 0.3
      max: 1.0
visualization:
  fig_size: [20, 14]
  font_size: 16

Writing config_filter.yaml


In [ ]:
%%writefile config_cluster.yaml
# config_cluster_dbscan.yaml
# Режим кластеризации графов зависимостей
input:
  gold: "/content/be_treebanks.pkl"                 # эталонные деревья (словарь по корпусам)
  # parser и parser_name не требуются в режиме cluster

output_dir: "/content/cluster_dbscan_output"

mode: "cluster"

# Параметры векторизации (Graph2Vec)
vectorization:
  method: "graph2vec"
  node_feature_mode: "dep_degree"   # dep_degree | dep_only | degree | constant
  dimensions: 128
  wl_iterations: 3
  epochs: 80
  workers: 2
  min_count: 2
  learning_rate: 0.025
  down_sampling: 0.0001

# Параметры кластеризации
clustering:
  # Один из методов: "dbscan" или "hdbscan"
  method: "dbscan"

  # Настройки DBSCAN
  dbscan:
    # Если eps не задан, он будет определён автоматически по графику k-расстояний
    # eps: 0.5
    min_samples: 10
    # metric: "cosine"          # по умолчанию euclidean
    # eps_quantile: 0.90       # если подбираем eps автоматически

  # Настройки HDBSCAN (будут использованы, если method = "hdbscan")
  hdbscan:
    min_cluster_size: 10
    # min_samples: 5
    # metric: "euclidean"

  # Метрики качества кластеризации (внутренние)
  compute_metrics: false
  # Если false, выводятся только базовые статистики (размер, распределение по датасетам)

# Визуализация (общие параметры для обоих режимов)
visualization:
  fig_size: [20, 14]
  font_size: 16

Writing config_cluster.yaml


In [ ]:
!conda run -n g2v37 python run_utility.py config_filter.yaml


CondaError: KeyboardInterrupt



In [ ]:
import pickle
with open('/content/be_treebanks.pkl', 'rb') as f:
    gold = pickle.load(f)
print('Эталон:', type(gold), len(gold), list(gold.keys()))

with open('/content/be_parser_res.pkl', 'rb') as f:
    parser = pickle.load(f)
print('Парсер:', type(parser), len(parser), list(parser.keys()))

Эталон: <class 'dict'> 5 ['taiga', 'poetry', 'gsd', 'pud', 'syntagrus']
Парсер: <class 'dict'> 5 ['natasha', 'udpipe', 'spacy', 'deeppavlov', 'stanza']


## Вторая часть

In [ ]:
%%writefile graph2vec_clustering.py
from __future__ import annotations

import argparse
import json
import os
import pickle
import random
import sys
import warnings
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from karateclub import Graph2Vec
from sklearn.cluster import AgglomerativeClustering, AffinityPropagation, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.metrics.pairwise import cosine_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize, StandardScaler
from tqdm import tqdm

try:
    import umap  # type: ignore
except Exception:
    umap = None

Span = Tuple[int, int]
DepsDict = Mapping[Span, Tuple[Span, str]]

ROOT_SPANS = {(-1, -1), (0, 0), None}


@dataclass(frozen=True)
class ExperimentConfig:
    seed: int = 42
    node_feature_mode: str = "dep_degree"  # dep_degree | dep_only | degree | constant
    keep_dependency_direction_as_feature: bool = True
    graph_undirected_for_graph2vec: bool = True
    dimensions: int = 128
    wl_iterations: int = 3
    epochs: int = 80
    workers: int = 2
    min_count: int = 2
    learning_rate: float = 0.025
    down_sampling: float = 0.0001
    clustering_k_min: int = 2
    clustering_k_max: int = 15
    dbscan_min_samples: int = 10
    dbscan_eps_quantiles: Tuple[float, ...] = (0.85, 0.90, 0.95, 0.975, 0.99)


def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def assert_environment() -> None:
    import scipy
    import sklearn
    import networkx
    import gensim
    import karateclub

    py = sys.version_info
    if not (py.major == 3 and py.minor == 7):
        raise RuntimeError(
            f"Нужен Python 3.7 для стабильного karateclub==1.3.3, сейчас: {sys.version}"
        )

    versions = {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "sklearn": sklearn.__version__,
        "networkx": networkx.__version__,
        "gensim": gensim.__version__,
        "karateclub": getattr(karateclub, "__version__", "unknown"),
    }
    print(json.dumps(versions, indent=2, ensure_ascii=False))

    assert versions["numpy"].startswith("1.21"), versions
    assert versions["scipy"].startswith("1.7"), versions
    assert versions["sklearn"].startswith("1.0"), versions
    assert versions["networkx"].startswith("2.6"), versions
    assert versions["gensim"].startswith("4.2"), versions


def load_pickle(path: Path) -> Any:
    with path.open("rb") as f:
        return pickle.load(f)


def normalize_span(x: Any) -> Span:
    if x is None:
        return (-1, -1)
    if isinstance(x, tuple) and len(x) == 2:
        return int(x[0]), int(x[1])
    if isinstance(x, list) and len(x) == 2:
        return int(x[0]), int(x[1])
    raise ValueError(f"Некорректный span: {x!r}")


def is_root_span(span: Any) -> bool:
    try:
        s = normalize_span(span)
    except Exception:
        return span is None
    return s in {(-1, -1), (0, 0)}


def validate_deps_dict(deps: DepsDict, sent_id: str) -> None:
    children = {normalize_span(ch) for ch in deps.keys()}
    if not children:
        raise ValueError(f"{sent_id}: пустой граф")

    roots = 0
    for child, value in deps.items():
        child = normalize_span(child)
        if not isinstance(value, (tuple, list)) or len(value) != 2:
            raise ValueError(f"{sent_id}: значение deps_dict должно быть (parent_span, relation), got {value!r}")
        parent, rel = value
        parent = normalize_span(parent)
        if is_root_span(parent):
            roots += 1
        elif parent not in children:
            raise ValueError(f"{sent_id}: parent {parent} for child {child} отсутствует среди узлов")
        if not isinstance(rel, str) or not rel:
            raise ValueError(f"{sent_id}: пустая relation label у child {child}")

    if roots != 1:
        warnings.warn(f"{sent_id}: ожидался 1 root, найдено {roots}")


def build_feature_for_span_graph(
    node: Span,
    parent: Span,
    relation: str,
    undirected_degree: int,
    out_degree: int,
    in_degree: int,
    cfg: ExperimentConfig,
) -> str:
    rel = "ROOT" if is_root_span(parent) else relation

    if cfg.node_feature_mode == "dep_degree":
        return f"rel={rel}|deg={undirected_degree}|in={in_degree}|out={out_degree}"
    if cfg.node_feature_mode == "dep_only":
        return f"rel={rel}"
    if cfg.node_feature_mode == "degree":
        return f"deg={undirected_degree}|in={in_degree}|out={out_degree}"
    if cfg.node_feature_mode == "constant":
        return "NODE"
    raise ValueError(f"Unknown node_feature_mode={cfg.node_feature_mode!r}")


def deps_to_graph(deps: DepsDict, sent_id: str, cfg: ExperimentConfig) -> nx.Graph:
    validate_deps_dict(deps, sent_id)

    directed = nx.DiGraph()
    for child_raw, (parent_raw, rel) in deps.items():
        child = normalize_span(child_raw)
        parent = normalize_span(parent_raw)
        directed.add_node(child)
        if not is_root_span(parent):
            directed.add_node(parent)
            directed.add_edge(parent, child, dep=rel)

    ordered_nodes = sorted(directed.nodes(), key=lambda x: (x[0], x[1]))
    mapping = {node: i for i, node in enumerate(ordered_nodes)}

    und = nx.Graph()
    for old_node in ordered_nodes:
        new_node = mapping[old_node]
        und.add_node(new_node, original_span=str(old_node))

    for u, v, data in directed.edges(data=True):
        und.add_edge(mapping[u], mapping[v], dep=data.get("dep", "dep"))

    und_degrees = dict(und.degree())
    in_degrees_old = dict(directed.in_degree())
    out_degrees_old = dict(directed.out_degree())

    child_to_parent_rel = {
        normalize_span(child): (normalize_span(parent), rel)
        for child, (parent, rel) in deps.items()
    }

    for old_node in ordered_nodes:
        parent, rel = child_to_parent_rel.get(old_node, ((-1, -1), "root"))
        new_node = mapping[old_node]
        feature = build_feature_for_span_graph(
            node=old_node,
            parent=parent,
            relation=rel,
            undirected_degree=und_degrees[new_node],
            in_degree=in_degrees_old.get(old_node, 0),
            out_degree=out_degrees_old.get(old_node, 0),
            cfg=cfg,
        )
        und.nodes[new_node]["feature"] = feature

    if und.number_of_nodes() == 0:
        raise ValueError(f"{sent_id}: graph has zero nodes after conversion")
    return und


def flatten_be_treebanks(be_treebanks: Mapping[str, Mapping[Any, DepsDict]], cfg: ExperimentConfig) -> Tuple[List[nx.Graph], pd.DataFrame]:
    graphs: List[nx.Graph] = []
    rows: List[Dict[str, Any]] = []

    for treebank_name, tb in be_treebanks.items():
        if not isinstance(tb, Mapping):
            raise TypeError(f"be_treebanks[{treebank_name!r}] должен быть mapping sent_id -> deps_dict")
        for sent_id, deps in tqdm(tb.items(), desc=f"graphs:{treebank_name}"):
            sid = str(sent_id)
            graph = deps_to_graph(deps, sid, cfg)
            graph.graph["sent_id"] = sid
            graph.graph["treebank"] = str(treebank_name)
            graphs.append(graph)
            rows.append(
                {
                    "graph_index": len(graphs) - 1,
                    "sent_id": sid,
                    "treebank": str(treebank_name),
                    "n_nodes": graph.number_of_nodes(),
                    "n_edges": graph.number_of_edges(),
                    "node_features": " ".join(sorted({str(v) for _, v in graph.nodes(data="feature")})),
                }
            )

    meta = pd.DataFrame(rows)
    if meta.empty:
        raise ValueError("Не найдено ни одного графа")
    return graphs, meta


def fit_graph2vec(graphs: Sequence[nx.Graph], cfg: ExperimentConfig) -> np.ndarray:
    model = Graph2Vec(
        wl_iterations=cfg.wl_iterations,
        attributed=True,
        dimensions=cfg.dimensions,
        workers=cfg.workers,
        down_sampling=cfg.down_sampling,
        epochs=cfg.epochs,
        learning_rate=cfg.learning_rate,
        min_count=cfg.min_count,
        seed=cfg.seed,
    )
    model.fit(list(graphs))
    emb = model.get_embedding()
    if not isinstance(emb, np.ndarray):
        emb = np.asarray(emb)
    if emb.shape[0] != len(graphs):
        raise RuntimeError(f"Graph2Vec returned {emb.shape[0]} embeddings for {len(graphs)} graphs")
    if not np.isfinite(emb).all():
        raise RuntimeError("Graph2Vec embeddings contain NaN/Inf")
    return emb.astype(np.float32)


def safe_internal_metrics(X: np.ndarray, labels: np.ndarray, metric: str = "euclidean") -> Dict[str, Any]:
    labels = np.asarray(labels)
    mask = labels != -1
    non_noise_labels = labels[mask]
    n_clusters = len(set(non_noise_labels.tolist()))
    result: Dict[str, Any] = {
        "n_clusters": int(n_clusters),
        "noise_count": int((labels == -1).sum()),
        "noise_ratio": float((labels == -1).mean()),
    }

    if n_clusters < 2 or mask.sum() <= n_clusters:
        result.update({"silhouette": None, "davies_bouldin": None, "calinski_harabasz": None})
        return result

    Xm = X[mask]
    lm = labels[mask]
    result["silhouette"] = float(silhouette_score(Xm, lm, metric=metric))
    result["davies_bouldin"] = float(davies_bouldin_score(Xm, lm))
    result["calinski_harabasz"] = float(calinski_harabasz_score(Xm, lm))
    return result


def external_metrics_if_available(labels: np.ndarray, y_true: Optional[Sequence[Any]]) -> Dict[str, Any]:
    if y_true is None:
        return {}
    y = np.asarray(y_true)
    return {
        "ARI": float(adjusted_rand_score(y, labels)),
        "NMI": float(normalized_mutual_info_score(y, labels)),
        "AMI": float(adjusted_mutual_info_score(y, labels)),
    }


def relabel_noise_aware(labels: np.ndarray) -> np.ndarray:
    labels = np.asarray(labels)
    mapping: Dict[Any, int] = {}
    next_id = 0
    out = np.full_like(labels, fill_value=-1, dtype=int)
    for i, lab in enumerate(labels):
        if lab == -1:
            continue
        if lab not in mapping:
            mapping[lab] = next_id
            next_id += 1
        out[i] = mapping[lab]
    return out


def candidate_k_values(n: int, cfg: ExperimentConfig) -> List[int]:
    kmax = min(cfg.clustering_k_max, max(2, int(np.sqrt(n))))
    return list(range(cfg.clustering_k_min, kmax + 1))


def run_clustering_suite(
    X_raw: np.ndarray,
    meta: pd.DataFrame,
    cfg: ExperimentConfig,
    y_true: Optional[Sequence[Any]] = None,
) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    results: List[Dict[str, Any]] = []
    label_sets: Dict[str, np.ndarray] = {}

    X_euclidean = StandardScaler().fit_transform(X_raw)
    X_cosine = normalize(X_raw)

    def add_result(name: str, X_eval: np.ndarray, labels: np.ndarray, metric: str) -> None:
        labels = relabel_noise_aware(labels)
        key = name
        label_sets[key] = labels
        row = {"method": name, "metric_space": metric}
        row.update(safe_internal_metrics(X_eval, labels, metric=("cosine" if metric == "cosine" else "euclidean")))
        row.update(external_metrics_if_available(labels, y_true))
        results.append(row)

    n = X_raw.shape[0]
    ks = candidate_k_values(n, cfg)

    for k in ks:
        km = KMeans(n_clusters=k, n_init=20, random_state=cfg.seed)
        add_result(f"kmeans_k={k}", X_euclidean, km.fit_predict(X_euclidean), "euclidean")

        agg = AgglomerativeClustering(n_clusters=k, linkage="ward")
        add_result(f"agglomerative_ward_k={k}", X_euclidean, agg.fit_predict(X_euclidean), "euclidean")

        agg_cos = AgglomerativeClustering(n_clusters=k, affinity="cosine", linkage="average")
        add_result(f"agglomerative_cosine_k={k}", X_cosine, agg_cos.fit_predict(X_cosine), "cosine")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        ap = AffinityPropagation(random_state=cfg.seed, max_iter=500, convergence_iter=30)
        try:
            add_result("affinity_propagation", X_euclidean, ap.fit_predict(X_euclidean), "euclidean")
        except Exception as e:
            results.append({"method": "affinity_propagation", "error": repr(e)})

    for metric_name, X_space in [("euclidean", X_euclidean), ("cosine", X_cosine)]:
        nn_metric = metric_name
        n_neighbors = min(cfg.dbscan_min_samples, n - 1)
        nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric=nn_metric)
        nbrs.fit(X_space)
        distances, _ = nbrs.kneighbors(X_space)
        kth = np.sort(distances[:, -1])
        for q in cfg.dbscan_eps_quantiles:
            eps = float(np.quantile(kth, q))
            db = DBSCAN(eps=eps, min_samples=cfg.dbscan_min_samples, metric=nn_metric)
            labels = db.fit_predict(X_space)
            method = f"dbscan_{metric_name}_eps_q={q:.3f}_eps={eps:.5f}"
            add_result(method, X_space, labels, metric_name)

    scores = pd.DataFrame(results)
    if "silhouette" in scores.columns:
        scores = scores.sort_values(
            by=["silhouette", "n_clusters", "noise_ratio"],
            ascending=[False, False, True],
            na_position="last",
        ).reset_index(drop=True)
    return scores, label_sets


def stability_by_seeds(
    graphs: Sequence[nx.Graph],
    base_cfg: ExperimentConfig,
    seeds: Sequence[int],
    cluster_method: str,
    max_graphs_for_speed: Optional[int] = None,
) -> pd.DataFrame:
    if max_graphs_for_speed is not None and len(graphs) > max_graphs_for_speed:
        rng = np.random.default_rng(base_cfg.seed)
        idx = np.sort(rng.choice(len(graphs), size=max_graphs_for_speed, replace=False))
        work_graphs = [graphs[i] for i in idx]
    else:
        work_graphs = list(graphs)

    labels_by_seed: Dict[int, np.ndarray] = {}
    rows: List[Dict[str, Any]] = []

    for seed in seeds:
        cfg = ExperimentConfig(**{**asdict(base_cfg), "seed": int(seed)})
        set_seed(cfg.seed)
        emb = fit_graph2vec(work_graphs, cfg)
        dummy_meta = pd.DataFrame({"graph_index": np.arange(len(work_graphs))})
        scores, labels = run_clustering_suite(emb, dummy_meta, cfg)
        if cluster_method == "best_by_silhouette":
            best_name = str(scores.iloc[0]["method"])
        else:
            best_name = cluster_method
        labels_by_seed[seed] = labels[best_name]
        row = scores[scores["method"] == best_name].iloc[0].to_dict()
        row["seed"] = seed
        row["selected_method"] = best_name
        rows.append(row)

    out = pd.DataFrame(rows)
    pairwise = []
    for i, s1 in enumerate(seeds):
        for s2 in seeds[i + 1 :]:
            pairwise.append(adjusted_rand_score(labels_by_seed[s1], labels_by_seed[s2]))
    out.attrs["pairwise_ari_mean"] = float(np.mean(pairwise)) if pairwise else None
    out.attrs["pairwise_ari_std"] = float(np.std(pairwise)) if pairwise else None
    return out


def plot_projection(X: np.ndarray, labels: np.ndarray, out_path: Path, title: str) -> None:
    if X.shape[0] < 3:
        return
    if umap is not None and X.shape[0] >= 20:
        reducer = umap.UMAP(n_components=2, random_state=42, metric="cosine", n_neighbors=15, min_dist=0.1)
        Z = reducer.fit_transform(X)
        method = "UMAP"
    else:
        Z = PCA(n_components=2, random_state=42).fit_transform(X)
        method = "PCA"

    plt.figure(figsize=(9, 7))
    plt.scatter(Z[:, 0], Z[:, 1], c='darkviolet', s=8, alpha=0.75)
    plt.title(f"{title} ({method})")
    plt.xlabel("dim 1")
    plt.ylabel("dim 2")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def save_outputs(
    out_dir: Path,
    cfg: ExperimentConfig,
    meta: pd.DataFrame,
    embeddings: np.ndarray,
    scores: pd.DataFrame,
    label_sets: Dict[str, np.ndarray],
) -> None:
    out_dir.mkdir(parents=True, exist_ok=True)

    with (out_dir / "config.json").open("w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, ensure_ascii=False, indent=2)

    np.save(out_dir / "graph2vec_embeddings.npy", embeddings)
    meta.to_csv(out_dir / "graph_metadata.csv", index=False)
    scores.to_csv(out_dir / "clustering_scores.csv", index=False)

    best_method = str(scores.iloc[0]["method"])
    best_labels = label_sets[best_method]
    labeled = meta.copy()
    labeled["cluster"] = best_labels
    labeled.to_csv(out_dir / "best_clustering.csv", index=False)

    all_labels = pd.DataFrame({name: labels for name, labels in label_sets.items()})
    all_labels.insert(0, "graph_index", meta["graph_index"].values)
    all_labels.to_csv(out_dir / "all_cluster_labels.csv", index=False)

    summary = (
        labeled.groupby("cluster")
        .agg(
            n=("sent_id", "count"),
            treebanks=("treebank", lambda x: dict(Counter(x))),
            n_nodes_mean=("n_nodes", "mean"),
            n_edges_mean=("n_edges", "mean"),
            examples=("sent_id", lambda x: list(x.head(10))),
        )
        .reset_index()
        .sort_values("n", ascending=False)
    )
    summary.to_csv(out_dir / "cluster_summary.csv", index=False)

    plot_projection(embeddings, best_labels, out_dir / "projection_best_clustering.png", best_method)

    print("\nЛучший метод по текущей сортировке:", best_method)
    print("\nTop-10 score table:")
    print(scores.head(10).to_string(index=False))
    print(f"\nФайлы сохранены в: {out_dir.resolve()}")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", type=Path, default=None, help="Path to be_treebanks.pkl (gold graphs)")
    parser.add_argument("--parser-file", type=Path, default=None, help="Path to pickle file with all parser results (be_parser_res.pkl)")
    parser.add_argument("--parser-name", type=str, default=None, help="Which parser to use (e.g., stanza, udpipe, spacy)")
    parser.add_argument("--out", type=Path, default=None, help="Output directory (auto-generated if not given)")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--node-feature-mode", default="dep_degree", choices=["dep_degree", "dep_only", "degree", "constant"])
    parser.add_argument("--dimensions", type=int, default=128)
    parser.add_argument("--wl-iterations", type=int, default=3)
    parser.add_argument("--epochs", type=int, default=80)
    parser.add_argument("--workers", type=int, default=2)
    parser.add_argument("--min-count", type=int, default=2)
    parser.add_argument("--k-min", type=int, default=2)
    parser.add_argument("--k-max", type=int, default=15)
    parser.add_argument("--dbscan-min-samples", type=int, default=10)
    parser.add_argument("--external-label", default=None, choices=[None, "treebank"])
    parser.add_argument("--skip-env-check", action="store_true")
    args = parser.parse_args()

    if not args.skip_env_check:
        assert_environment()

    # ----------- Определяем источник данных -----------
    if args.parser_file and args.parser_name:
        log(f"Режим парсера: {args.parser_name}")
        all_parser_data = load_pickle(args.parser_file)
        if args.parser_name not in all_parser_data:
            raise ValueError(f"Парсер '{args.parser_name}' не найден в файле. Доступные: {list(all_parser_data.keys())}")
        data = all_parser_data[args.parser_name]
        source_name = args.parser_name
    elif args.input:
        log("Режим gold-эталона")
        data = load_pickle(args.input)
        source_name = "gold"
    else:
        raise ValueError("Укажите либо --input (gold), либо --parser-file и --parser-name")

    out_dir = args.out or Path(f"graph2vec_{source_name}_results")

    cfg = ExperimentConfig(
        seed=args.seed,
        node_feature_mode=args.node_feature_mode,
        dimensions=args.dimensions,
        wl_iterations=args.wl_iterations,
        epochs=args.epochs,
        workers=args.workers,
        min_count=args.min_count,
        clustering_k_min=args.k_min,
        clustering_k_max=args.k_max,
        dbscan_min_samples=args.dbscan_min_samples,
    )

    set_seed(cfg.seed)
    log("Загрузка данных")
    # data уже загружена, содержит {treebank: OrderedDict(sent_id -> deps)}

    log("Построение графов")
    checkpoints_dir = out_dir / "checkpoints"
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    graphs_path = checkpoints_dir / "chk_1_graphs.pkl"
    if graphs_path.exists():
        log("Загружаем graphs из checkpoint")
        graphs, meta = load_checkpoint(graphs_path)
    else:
        graphs, meta = flatten_be_treebanks(data, cfg)
        save_checkpoint((graphs, meta), graphs_path)

    print(f"Графов: {len(graphs)}")
    print(meta[["n_nodes", "n_edges"]].describe().to_string())
    print("Treebanks:", meta["treebank"].value_counts().to_dict())

    log("Graph2Vec")
    emb_path = checkpoints_dir / "chk_2_embeddings.pkl"
    if emb_path.exists():
        log("Загружаем embeddings из checkpoint")
        embeddings = load_checkpoint(emb_path)
    else:
        embeddings = fit_graph2vec(graphs, cfg)
        np.save(out_dir / "embeddings.npy", embeddings)
        save_checkpoint(embeddings, emb_path)

    y_true = meta[args.external_label].values if args.external_label else None

    log("Кластеризация")
    clust_path = checkpoints_dir / "chk_3_clusters.pkl"
    if clust_path.exists():
        log("Загружаем clusters из checkpoint")
        scores, label_sets = load_checkpoint(clust_path)
    else:
        scores, label_sets = run_clustering_suite(embeddings, meta, cfg, y_true=y_true)
        save_checkpoint((scores, label_sets), clust_path)

    log("Сохранение")
    save_outputs(out_dir, cfg, meta, embeddings, scores, label_sets)


def log(msg):
    print(f"[LOG {time.strftime('%H:%M:%S')}] {msg}")


def save_checkpoint(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'wb') as f:
        pickle.dump(obj, f)
    log(f"Checkpoint сохранён: {path}")


def load_checkpoint(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


if __name__ == "__main__":
    main()

Writing graph2vec_clustering.py



### 2. Запуск pipeline

 Загрузите/сохраните файл `be_treebanks.pkl` в `/content/be_treebanks.pkl`.

 Если объект `be_treebanks` уже есть в памяти основного Colab-ноутбука, сохраните его так:

 ```python
 import pickle
 with open('/content/be_treebanks.pkl', 'wb') as f:
     pickle.dump(be_treebanks, f)
 ```


In [ ]:
# --- Colab cell 4: запуск через совместимое conda-env ---
!conda run -n g2v37 python graph2vec_clustering.py \
    --input /content/be_treebanks.pkl \
    --out /content/graph2vec_gold_results \
    --seed 42 \
    --node-feature-mode dep_degree \
    --dimensions 128 \
    --wl-iterations 3 \
    --epochs 10 \
    --workers 2 \
    --min-count 2 \
    --k-min 2 \
    --k-max 15 \
    --dbscan-min-samples 10 \
    --external-label treebank


CondaError: KeyboardInterrupt



In [ ]:
!conda run -n g2v37 python graph2vec_clustering.py \
    --parser-file /content/be_parser_res.pkl \
    --parser-name stanza \
    --out /content/graph2vec_stanza_results \
    --seed 42 \
    --node-feature-mode dep_degree \
    --dimensions 128 \
    --wl-iterations 3 \
    --epochs 80 \
    --workers 2 \
    --min-count 2 \
    --k-min 2 \
    --k-max 15 \
    --dbscan-min-samples 10 \
    --external-label treebank


CondaError: KeyboardInterrupt



In [ ]:
import shutil
from google.colab import files

folder_path = "/content/graph2vec_deeppavlov_results"
archive_name = shutil.make_archive('deeppavlov', 'zip', folder_path)
files.download(archive_name)

In [ ]:
folder_path = "/content/graph2vec_stanza_results"
archive_name = shutil.make_archive('stanza', 'zip', folder_path)
files.download(archive_name)

### 3. Просмотр результатов

In [ ]:
import pandas as pd
from IPython.display import display, Image

scores = pd.read_csv('/content/graph2vec_gold_results/clustering_scores.csv')
display(scores.head(20))

summary = pd.read_csv('/content/graph2vec_gold_results/cluster_summary.csv')
display(summary.head(20))

Image('/content/graph2vec_gold_results/projection_best_clustering.png')

Вариант, если доступны исходные gold-токены с UPOS/morph

 Если есть не только `be_treebanks`, а исходные gold-предложения с токенами, используйте node feature
 из лингвистических признаков. Ниже шаблон адаптера. Его нужно подстроить под реальные классы токенов.


In [ ]:
def build_gold_graph_from_sentence(sentence):
    """
    Шаблон. Ожидается, что sentence.tokens содержит gold-токены с полями:
    id, parent_id, relation, text, upos, feats/lemma.

    Возвращает undirected NetworkX graph с node['feature'], пригодный для KarateClub Graph2Vec.
    Parser outputs здесь не нужны и не используются.
    """
    import networkx as nx

    directed = nx.DiGraph()
    id_to_node = {}

    for tok in sentence.tokens:
        # Пропускаем UD empty nodes / multiword tokens при необходимости.
        if isinstance(tok.id, str) and ("." in tok.id or "-" in tok.id):
            continue
        tid = int(tok.id)
        id_to_node[tid] = tid
        upos = getattr(tok, "upos", "X") or "X"
        lemma = getattr(tok, "lemma", "_") or "_"
        feats = getattr(tok, "feats", None) or "_"
        directed.add_node(tid, feature=f"upos={upos}|feats={feats}")

    for tok in sentence.tokens:
        if isinstance(tok.id, str) and ("." in tok.id or "-" in tok.id):
            continue
        tid = int(tok.id)
        pid = int(tok.parent_id)
        rel = getattr(tok, "relation", "dep") or "dep"
        # Добавляем relation к feature зависимого, чтобы edge label не исчезал в Graph2Vec.
        directed.nodes[tid]["feature"] = directed.nodes[tid]["feature"] + f"|deprel={rel if pid != 0 else 'ROOT'}"
        if pid != 0:
            directed.add_edge(pid, tid, dep=rel)

    # Graph2Vec/KarateClub практически работает с неориентированным WL.
    und = nx.Graph()
    mapping = {old: i for i, old in enumerate(sorted(directed.nodes()))}
    for old, new in mapping.items():
        und.add_node(new, feature=directed.nodes[old]["feature"], original_token_id=old)
    for u, v, data in directed.edges(data=True):
        und.add_edge(mapping[u], mapping[v], dep=data.get("dep", "dep"))
    return und

# Эксперимент

In [ ]:
%%writefile experiment.py
"""
Эксперимент: обнаружение шумовых предложений через кластеризацию gold‑деревьев
и их ранжирование по метрикам всех парсеров.
"""

import argparse
import json
import os
import pickle
import random
import sys
import warnings
import time
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from karateclub import Graph2Vec
from sklearn.cluster import AgglomerativeClustering, AffinityPropagation, DBSCAN, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    normalized_mutual_info_score,
    silhouette_score,
)
from sklearn.metrics.pairwise import cosine_distances
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize, StandardScaler
from tqdm import tqdm
from pathlib import Path

# Импортируем функции из созданных модулей
from run_utility import load_datasets, load_pickle, flatten_treebank_dict
from graph2vec_clustering import (
    ExperimentConfig,
    fit_graph2vec,
    flatten_be_treebanks,
    set_seed
)
from dep_metrics import (
    networkx_formatter,
    new_wl_ker1,
    graph_edit_distance_with_labels,
    structural_distortion_score,
    find_root,
)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
# Опционально HDBSCAN:
# from hdbscan import HDBSCAN   # или from sklearn.cluster import HDBSCAN

# ---------------------------------------------------------------------------
# Конфигурация эксперимента
# ---------------------------------------------------------------------------
GOLD_PKL = "/content/be_treebanks.pkl"          # эталонные деревья
PARSER_PKL = "/content/be_parser_res.pkl"       # результаты всех парсеров
OUTPUT_DIR = Path("/content/noise_ranking_experiment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Параметры векторизации и кластеризации (меняйте при необходимости)
SEED = 42
CLUSTER_METHOD = "dbscan"      # или "hdbscan"
DBSCAN_EPS_QUANTILE = 0.75     # для автоматического подбора eps
DBSCAN_MIN_SAMPLES = 10
# Для HDBSCAN:
# HDBSCAN_MIN_CLUSTER_SIZE = 10

# ---------------------------------------------------------------------------
# 1. Загрузка данных
# ---------------------------------------------------------------------------
gold_raw = load_datasets({"input": {"gold": GOLD_PKL, "parser": None}})[0]
parser_raw = load_datasets({"input": {"gold": GOLD_PKL, "parser": PARSER_PKL,
                                      "parser_name": "stanza"}})[1]
# parser_raw содержит словарь {treebank: {sent_id: deps}} для stanza,
# но у нас общий файл со всеми парсерами; загрузим отдельно.
all_parsers = load_pickle(PARSER_PKL)  # {parser_name: {treebank: ...}}
parser_names = list(all_parsers.keys())

# Создаём плоские словари для gold и для каждого парсера
gold_flat = flatten_treebank_dict(gold_raw)   # {sent_id: deps}
parsers_flat = {}
for pname in parser_names:
    parsers_flat[pname] = flatten_treebank_dict(all_parsers[pname])

# ---------------------------------------------------------------------------
# 2. Кластеризация gold‑деревьев
# ---------------------------------------------------------------------------
g2v_cfg = ExperimentConfig(
    seed=SEED,
    node_feature_mode="dep_degree",
    dimensions=128,
    wl_iterations=3,
    epochs=80,
    workers=2,
    min_count=2,
    learning_rate=0.025,
    down_sampling=0.0001,
)
set_seed(SEED)

# Построение графов и эмбеддингов
graphs, meta = flatten_be_treebanks(gold_raw, g2v_cfg)
embeddings = fit_graph2vec(graphs, g2v_cfg)

# Стандартизация
X = StandardScaler().fit_transform(embeddings)

# Кластеризация
if CLUSTER_METHOD == "dbscan":
    from sklearn.neighbors import NearestNeighbors
    n_neighbors = min(DBSCAN_MIN_SAMPLES, X.shape[0] - 1)
    nbrs = NearestNeighbors(n_neighbors=n_neighbors, metric="euclidean").fit(X)
    distances, _ = nbrs.kneighbors(X)
    kth = np.sort(distances[:, -1])
    eps = float(np.quantile(kth, DBSCAN_EPS_QUANTILE))
    model = DBSCAN(eps=eps, min_samples=DBSCAN_MIN_SAMPLES, metric="euclidean")
elif CLUSTER_METHOD == "hdbscan":
    from hdbscan import HDBSCAN
    model = HDBSCAN(min_cluster_size=10)
labels = model.fit_predict(X)

# Шумовые предложения
meta["cluster"] = labels
noise_mask = labels == -1
noise_sent_ids = meta.loc[noise_mask, "sent_id"].tolist()
print(f"Найдено шумовых предложений: {len(noise_sent_ids)}")

noise_sent_ids_path = "noise_sent_ids.pkl"
with open(noise_sent_ids_path, "wb") as f:
    pickle.dump(noise_sent_ids, f)
print(f"Список шумовых предложений сохранён в {noise_sent_ids_path}")


def compute_las_uas(gold_deps, parser_deps):
    """
    Вычисляет UAS (Unlabeled Attachment Score) и LAS (Labeled Attachment Score)
    для пары словарей зависимостей {token: (head, rel)}.
    Возвращает (uas, las). Все токены gold считаются за эталон.
    """
    gold_tokens = set(gold_deps.keys())
    parser_tokens = set(parser_deps.keys())
    common_tokens = gold_tokens & parser_tokens

    correct_heads = 0
    correct_heads_labels = 0
    for token in common_tokens:
        gold_head, gold_rel = gold_deps[token]
        parser_head, parser_rel = parser_deps[token]
        if gold_head == parser_head:
            correct_heads += 1
            if gold_rel == parser_rel:
                correct_heads_labels += 1

    total_gold = len(gold_tokens)
    if total_gold == 0:
        return 0.0, 0.0
    uas = correct_heads / total_gold
    las = correct_heads_labels / total_gold
    return uas, las




# ===== 2.1 Сохранение общей информации о кластеризации =====
# метод и параметры
clustering_info = {
    "method": CLUSTER_METHOD,
    "seed": SEED,
}
if CLUSTER_METHOD == "dbscan":
    clustering_info["eps"] = eps
    clustering_info["min_samples"] = DBSCAN_MIN_SAMPLES
elif CLUSTER_METHOD == "hdbscan":
    clustering_info["min_cluster_size"] = HDBSCAN_MIN_CLUSTER_SIZE

# подсчёт кластеров (без учёта шума)
n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))
n_total = len(labels)
clustering_info.update({
    "n_clusters": n_clusters,
    "n_noise": n_noise,
    "noise_ratio": n_noise / n_total if n_total else 0,
    "n_total": n_total,
})
pd.DataFrame([clustering_info]).to_csv(OUTPUT_DIR / "clustering_parameters.csv", index=False)
print(f"Параметры кластеризации сохранены в {OUTPUT_DIR / 'clustering_parameters.csv'}")

# ===== 2.2 Сводка по кластерам (размер, распределение по датасетам, примеры) =====
meta["cluster"] = labels
summary = (
    meta.groupby("cluster")
    .agg(
        n=("sent_id", "count"),
        treebanks=("treebank", lambda x: dict(pd.Series(x).value_counts())),
        n_nodes_mean=("n_nodes", "mean"),
        n_edges_mean=("n_edges", "mean"),
        examples=("sent_id", lambda x: list(x.head(10))),
    )
    .reset_index()
)
summary = summary.sort_values("n", ascending=False)
summary.to_csv(OUTPUT_DIR / "cluster_summary.csv", index=False)
print(f"Сводка по кластерам сохранена в {OUTPUT_DIR / 'cluster_summary.csv'}")

# ===== 2.3 Визуализация проекции эмбеддингов =====
# Импорт (если ещё не сделан) в начало скрипта:
from graph2vec_clustering import plot_projection
plot_projection(embeddings, labels, OUTPUT_DIR / "projection.png", title=f"Clustering ({CLUSTER_METHOD})")
print(f"Проекция сохранена в {OUTPUT_DIR / 'projection.png'}")

# ---------------------------------------------------------------------------
# 3. Вычисление метрик для шумовых предложений по всем парсерам
# ---------------------------------------------------------------------------
records = []
for sid in noise_sent_ids:
    if sid not in gold_flat:
        continue
    gold_deps = gold_flat[sid]
    for pname in parser_names:
        if sid not in parsers_flat[pname]:
            continue
        parser_deps = parsers_flat[pname][sid]
        G_gold = networkx_formatter(gold_deps, nx.DiGraph())
        G_parser = networkx_formatter(parser_deps, nx.DiGraph())

        # WL similarity
        wl_sim = new_wl_ker1([G_gold], [G_parser], h=3)

        uas_val, las_val = compute_las_uas(gold_deps, parser_deps)

        # GED
        # ged_val = graph_edit_distance_with_labels(G_gold, G_parser, timeout=5)

        # Structural
        gold_root = find_root(gold_deps)
        parser_root = find_root(parser_deps)
        struct_val = structural_distortion_score(
            gold_deps, parser_deps, gold_root, parser_root
        )

        records.append({
            "sent_id": sid,
            "parser": pname,
            "wl_sim": wl_sim,
            "ged": 0,
            "structural": struct_val,
            "uas": uas_val,
            "las": las_val,
        })

df = pd.DataFrame(records)

# ---------------------------------------------------------------------------
# 4. Аггрегация и ранжирование
# ---------------------------------------------------------------------------
# Для каждого предложения вычислим средние/медианные значения
agg = df.groupby("sent_id").agg(
    avg_wl_sim=("wl_sim", "mean"),
    avg_ged=("ged", "mean"),
    avg_structural=("structural", "mean"),
    parsers_available=("parser", "count")
).reset_index()

# Ранжирование: чем хуже метрики, тем выше.
# Для wl_sim: чем меньше, тем хуже; превращаем в "ошибку" = 1 - avg_wl_sim.
agg["wl_error"] = 1 - agg["avg_wl_sim"]
# Для GED и structural: чем больше, тем хуже.

# Составной индекс: можно использовать средний ранг по трём метрикам.
from scipy.stats import rankdata
agg["rank_wl"] = rankdata(-agg["wl_error"])      # больше ошибка -> меньший ранг (1 - лучший)
agg["rank_ged"] = rankdata(agg["avg_ged"])
agg["rank_struct"] = rankdata(agg["avg_structural"])
agg["mean_rank"] = (agg["rank_wl"] + agg["rank_ged"] + agg["rank_struct"]) / 3.0
agg = agg.sort_values("mean_rank", ascending=False)  # самые плохие сверху

# Сохраняем
agg.to_csv(OUTPUT_DIR / "noise_ranking.csv", index=False)
df.to_csv(OUTPUT_DIR / "noise_detailed_metrics.csv", index=False)

print("Рейтинг шумовых предложений (первые 10):")
print(agg.head(10)[["sent_id", "avg_wl_sim", "avg_ged", "avg_structural", "mean_rank"]])

Overwriting experiment.py


In [ ]:
!conda run -n g2v37 python experiment.py

Найдено шумовых предложений: 616
Список шумовых предложений сохранён в noise_sent_ids.pkl
Параметры кластеризации сохранены в /content/noise_ranking_experiment/clustering_parameters.csv
Сводка по кластерам сохранена в /content/noise_ranking_experiment/cluster_summary.csv
Проекция сохранена в /content/noise_ranking_experiment/projection.png
Рейтинг шумовых предложений (первые 10):
                             sent_id  avg_wl_sim  ...  avg_structural   mean_rank
165            2010Ekstremizm.xml_58    0.994088  ...        0.322752  497.833333
387   2018Poseshchenie_muzeya.xml_16    0.994875  ...        0.261123  495.833333
374  2018Poseshchenie_muzeya.xml_105    0.994060  ...        0.263925  489.166667
491                             7183    0.993243  ...        0.343197  488.833333
263              2012Kolchak.xml_106    0.995014  ...        0.224058  483.833333
208            2011Formula-1.xml_315    0.995545  ...        0.212115  480.166667
543                        test-s531    0.9

In [ ]:
+import pickle

# Путь к сохранённому файлу (должен совпадать с OUTPUT_DIR в скрипте)
noise_file = "/content/noise_sent_ids.pkl"

with open(noise_file, "rb") as f:
    noise_sent_ids = pickle.load(f)

print(f"Загружено {len(noise_sent_ids)} шумовых sent_id")
print(noise_sent_ids[:5])  # пример

Загружено 616 шумовых sent_id
['2023', '2027', '2125', '3003', '6165']


# Очередной эксперимент

In [ ]:
%%writefile cluster_short_sentences.py
"""
Скрипт для кластеризации эталонных деревьев на предложениях длиной менее 20 токенов.
Использует Graph2Vec для получения эмбеддингов и DBSCAN для кластеризации.
Результаты сохраняются в CSV-файлы.
"""

import os
import sys
import pickle
import argparse
import numpy as np
import pandas as pd
from collections import defaultdict
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from pathlib import Path

# Добавляем путь к модулям, если необходимо
sys.path.append('/content/parser_stat')  # или текущая директория

from graph2vec_clustering import ExperimentConfig, flatten_be_treebanks, fit_graph2vec, deps_to_graph


def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def remove_punct(deps):
    return {t: (h, r) for t, (h, r) in deps.items() if r != 'punct'}


def main():
    parser = argparse.ArgumentParser(description='Кластеризация эталонных деревьев для коротких предложений')
    parser.add_argument('--gold', type=str, default='/content/be_treebanks.pkl', help='Путь к файлу с эталонными деревьями')
    parser.add_argument('--parser-file', type=str, default='/content/be_parser_res.pkl', help='Путь к файлу с результатами парсеров')
    parser.add_argument('--out-dir', type=str, default='/content/cluster_short_results', help='Директория для сохранения результатов')
    parser.add_argument('--max-length', type=int, default=20, help='Максимальная длина предложения (после удаления пунктуации)')
    parser.add_argument('--dimensions', type=int, default=128)
    parser.add_argument('--wl-iterations', type=int, default=3)
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--min-samples', type=int, default=3)
    parser.add_argument('--eps-quantile', type=float, default=0.90)
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()

    # Загрузка данных
    be_treebanks = load_pickle(args.gold)
    all_parsers = load_pickle(args.parser_file)
    parser_names = list(all_parsers.keys())
    treebank_names = list(be_treebanks.keys())

    os.makedirs(args.out_dir, exist_ok=True)

    # Конфигурация Graph2Vec
    cfg = ExperimentConfig(
        seed=args.seed,
        node_feature_mode='dep_degree',
        dimensions=args.dimensions,
        wl_iterations=args.wl_iterations,
        epochs=args.epochs,
        workers=2,
        min_count=2,
        learning_rate=0.025,
        down_sampling=0.0001
    )

    results = {}

    for parser in parser_names:
        print(f"\nОбработка парсера {parser}...")
        # Собираем короткие предложения для данного парсера
        short_sids = []
        for dataset in treebank_names:
            if dataset not in all_parsers[parser]:
                continue
            gold_dict = be_treebanks[dataset]
            parser_dict = all_parsers[parser][dataset]
            common = set(gold_dict.keys()) & set(parser_dict.keys())
            for sid in common:
                gold_deps = remove_punct(gold_dict[sid])
                if len(gold_deps) < args.max_length:
                    short_sids.append((sid, dataset))
        if not short_sids:
            print(f"  Нет коротких предложений для {parser}")
            continue

        # Строим графы для эталонных деревьев
        graphs = []
        meta_rows = []
        for sid, dataset in short_sids:
            deps = remove_punct(be_treebanks[dataset][sid])
            G = deps_to_graph(deps, str(sid), cfg)
            graphs.append(G)
            meta_rows.append({
                'sent_id': sid,
                'dataset': dataset,
                'n_nodes': G.number_of_nodes()
            })
        meta = pd.DataFrame(meta_rows)

        # Получаем эмбеддинги
        embeddings = fit_graph2vec(graphs, cfg)

        # Стандартизация
        scaler = StandardScaler()
        X = scaler.fit_transform(embeddings)

        # Подбор eps
        from sklearn.neighbors import NearestNeighbors
        k = min(args.min_samples, X.shape[0] - 1)
        if k < 1:
            print(f"  Слишком мало предложений для {parser}, пропускаем")
            continue
        nbrs = NearestNeighbors(n_neighbors=k, metric='euclidean').fit(X)
        distances, _ = nbrs.kneighbors(X)
        kth_dist = np.sort(distances[:, -1])
        eps = np.percentile(kth_dist, args.eps_quantile * 100)

        # DBSCAN
        db = DBSCAN(eps=eps, min_samples=args.min_samples, metric='euclidean')
        labels = db.fit_predict(X)

        # Сохраняем результаты
        meta['cluster'] = labels
        meta['eps'] = eps
        out_file = os.path.join(args.out_dir, f'clusters_{parser}_short.csv')
        meta.to_csv(out_file, index=False)

        results[parser] = {
            'n_clusters': len(set(labels)) - (1 if -1 in labels else 0),
            'noise_count': int(np.sum(labels == -1)),
            'total': len(labels),
            'eps': eps
        }
        print(f"  {parser}: кластеров {results[parser]['n_clusters']}, шума {results[parser]['noise_count']}")

    # Сохраняем сводку
    summary_df = pd.DataFrame.from_dict(results, orient='index')
    summary_df.to_csv(os.path.join(args.out_dir, 'summary_short.csv'))
    print("\nГотово.")


if __name__ == '__main__':
    main()

Overwriting cluster_short_sentences.py


In [ ]:
%%writefile cluster_pairwise_distance.py
"""
Скрипт для кластеризации на основе расстояния между парами векторов
(эталонный и предсказанный) для каждого синтаксического анализатора.
Использует Graph2Vec для получения эмбеддингов эталонных и предсказанных деревьев,
затем вычисляет комбинированное расстояние между предложениями и применяет DBSCAN.
Результаты (кластеры + метрики UAS/LAS/WL и структурные критерии) сохраняются в CSV.
"""

import os
import sys
import pickle
import argparse
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.neighbors import NearestNeighbors
from pathlib import Path

sys.path.append('/content/parser_stat')

from graph2vec_clustering import ExperimentConfig, fit_graph2vec, deps_to_graph
from dep_metrics import new_wl_ker1, find_root, tree_depth_and_branching


def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def remove_punct(deps):
    return {t: (h, r) for t, (h, r) in deps.items() if r != 'punct'}


def compute_las_uas(gold_deps, parser_deps):
    """Вычисляет UAS и LAS для двух словарей."""
    gold_tokens = set(gold_deps.keys())
    parser_tokens = set(parser_deps.keys())
    common = gold_tokens & parser_tokens
    correct_heads = 0
    correct_labels = 0
    for t in common:
        gh, gl = gold_deps[t]
        ph, pl = parser_deps[t]
        if gh == ph:
            correct_heads += 1
            if gl == pl:
                correct_labels += 1
    total = len(gold_tokens)
    if total == 0:
        return 0.0, 0.0
    return correct_heads / total, correct_labels / total


def compute_structural_criteria(gold_deps, parser_deps):
    """
    Вычисляет критерии структурных ошибок для пары деревьев.
    Возвращает словарь с ключами: wrong_root, parent_ratio, depth_diff,
    branch_diff, reversed, depth_shift_ratio, SimpleStruct.
    """
    gold_root = find_root(gold_deps)
    parser_root = find_root(parser_deps)
    gold_tokens = set(gold_deps.keys())
    parser_tokens = set(parser_deps.keys())
    common = gold_tokens & parser_tokens

    wrong_root = 1 if gold_root != parser_root else 0

    if not common:
        parent_ratio = 1.0
        depth_shift_ratio = 1.0
        reversed_flag = 0
    else:
        changed_parent = sum(1 for t in common if gold_deps[t][0] != parser_deps[t][0])
        parent_ratio = changed_parent / len(common)

        # depth_shift_ratio
        def compute_depths(deps, root):
            import networkx as nx
            G = nx.DiGraph()
            G.add_nodes_from(deps.keys())
            for t, (h, _) in deps.items():
                if h != (-1, -1):
                    G.add_edge(h, t)
            depths = {}
            if root in G:
                queue = [(root, 0)]
                visited = {root}
                while queue:
                    node, d = queue.pop(0)
                    depths[node] = d
                    for child in G.successors(node):
                        if child not in visited:
                            visited.add(child)
                            queue.append((child, d + 1))
            return depths
        gold_depths = compute_depths(gold_deps, gold_root)
        parser_depths = compute_depths(parser_deps, parser_root)
        depth_shift = sum(1 for t in common if abs(gold_depths.get(t, 0) - parser_depths.get(t, 0)) > 1)
        depth_shift_ratio = depth_shift / len(common)

        # reversed
        gold_edges = {(h, t) for t, (h, _) in gold_deps.items() if h != (-1, -1)}
        parser_edges = {(h, t) for t, (h, _) in parser_deps.items() if h != (-1, -1)}
        reversed_flag = 1 if any((v, u) in parser_edges for (u, v) in gold_edges) else 0

    gold_depth, gold_branch = tree_depth_and_branching(gold_deps, gold_root)
    parser_depth, parser_branch = tree_depth_and_branching(parser_deps, parser_root)
    depth_diff = abs(gold_depth - parser_depth) / max(gold_depth, parser_depth, 1)
    branch_diff = abs(gold_branch - parser_branch) / max(gold_branch, parser_branch, 1e-6)

    SimpleStruct = (parent_ratio + depth_diff + branch_diff + depth_shift_ratio + wrong_root + reversed_flag) / 6.0

    return {
        'wrong_root': wrong_root,
        'parent_ratio': parent_ratio,
        'depth_diff': depth_diff,
        'branch_diff': branch_diff,
        'reversed': reversed_flag,
        'depth_shift_ratio': depth_shift_ratio,
        'SimpleStruct': SimpleStruct
    }


def deps_to_graph_allow_missing(deps, sent_id, cfg):
    """
    Преобразует словарь зависимостей в граф, но если родитель отсутствует среди узлов,
    добавляет родителя как изолированный узел (чтобы избежать ошибки валидации).
    """
    import networkx as nx
    from graph2vec_clustering import normalize_span, is_root_span

    nodes = set(deps.keys())
    for child, (head, rel) in deps.items():
        if head != (-1, -1) and head != 0:
            nodes.add(head)
    directed = nx.DiGraph()
    for node in nodes:
        directed.add_node(node)
    for child, (head, rel) in deps.items():
        if is_root_span(head):
            continue
        if head in nodes:
            directed.add_edge(head, child, dep=rel)
        else:
            continue

    ordered_nodes = sorted(directed.nodes(), key=lambda x: (x[0], x[1]))
    mapping = {node: i for i, node in enumerate(ordered_nodes)}
    und = nx.Graph()
    for old_node in ordered_nodes:
        new_node = mapping[old_node]
        und.add_node(new_node, original_span=str(old_node))
    for u, v, data in directed.edges(data=True):
        und.add_edge(mapping[u], mapping[v], dep=data.get("dep", "dep"))
    und_degrees = dict(und.degree())
    for old_node in ordered_nodes:
        new_node = mapping[old_node]
        # Простой признак – только степень
        und.nodes[new_node]["feature"] = f"deg={und_degrees[new_node]}"
    return und


def main():
    parser = argparse.ArgumentParser(description='Кластеризация на основе расстояния между парами векторов')
    parser.add_argument('--gold', type=str, default='/content/be_treebanks.pkl', help='Путь к файлу с эталонными деревьями')
    parser.add_argument('--parser-file', type=str, default='/content/be_parser_res.pkl', help='Путь к файлу с результатами парсеров')
    parser.add_argument('--out-dir', type=str, default='/content/cluster_pairwise_results', help='Директория для сохранения результатов')
    parser.add_argument('--max-length', type=int, default=None, help='Максимальная длина предложения (если не задано, все предложения)')
    parser.add_argument('--dimensions', type=int, default=128)
    parser.add_argument('--wl-iterations', type=int, default=3)
    parser.add_argument('--epochs', type=int, default=10)
    parser.add_argument('--min-samples', type=int, default=5)
    parser.add_argument('--eps-quantile', type=float, default=0.70)
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()

    be_treebanks = load_pickle(args.gold)
    all_parsers = load_pickle(args.parser_file)
    parser_names = list(all_parsers.keys())
    treebank_names = list(be_treebanks.keys())

    os.makedirs(args.out_dir, exist_ok=True)

    cfg = ExperimentConfig(
        seed=args.seed,
        node_feature_mode='dep_degree',
        dimensions=args.dimensions,
        wl_iterations=args.wl_iterations,
        epochs=args.epochs,
        workers=2,
        min_count=2,
        learning_rate=0.025,
        down_sampling=0.0001
    )

    results = {}

    for parser in parser_names:
        print(f"\nОбработка парсера {parser}...")
        sids = []
        for dataset in treebank_names:
            if dataset not in all_parsers[parser]:
                continue
            gold_dict = be_treebanks[dataset]
            parser_dict = all_parsers[parser][dataset]
            common = set(gold_dict.keys()) & set(parser_dict.keys())
            for sid in common:
                gold_deps = remove_punct(gold_dict[sid])
                if args.max_length is not None and len(gold_deps) >= args.max_length:
                    continue
                if sid not in parser_dict:
                    continue
                sids.append((sid, dataset))
        if not sids:
            print(f"  Нет подходящих предложений для {parser}")
            continue

        gold_graphs = []
        parser_graphs = []
        valid_sids = []
        # Для вычисления метрик сохраним также deps
        gold_deps_list = []
        parser_deps_list = []

        for sid, dataset in sids:
            try:
                gold_deps = remove_punct(be_treebanks[dataset][sid])
                parser_deps = remove_punct(all_parsers[parser][dataset][sid])
                Gg = deps_to_graph_allow_missing(gold_deps, str(sid), cfg)
                Gp = deps_to_graph_allow_missing(parser_deps, str(sid), cfg)
                gold_graphs.append(Gg)
                parser_graphs.append(Gp)
                valid_sids.append((sid, dataset))
                gold_deps_list.append(gold_deps)
                parser_deps_list.append(parser_deps)
            except Exception as e:
                print(f"  Пропускаем {sid} из-за ошибки: {e}")
                continue

        if not gold_graphs:
            print(f"  Не удалось построить графы для {parser}")
            continue

        # Эмбеддинги
        all_graphs = gold_graphs + parser_graphs
        embeddings = fit_graph2vec(all_graphs, cfg)
        n_gold = len(gold_graphs)
        gold_embs = embeddings[:n_gold]
        parser_embs = embeddings[n_gold:]

        # Расстояния
        dist_gold = euclidean_distances(gold_embs)
        dist_parser = euclidean_distances(parser_embs)
        if dist_gold.max() > 0:
            dist_gold = dist_gold / dist_gold.max()
        if dist_parser.max() > 0:
            dist_parser = dist_parser / dist_parser.max()
        combined_dist = dist_gold + dist_parser

        # DBSCAN
        k = min(args.min_samples, combined_dist.shape[0] - 1)
        if k < 1:
            print(f"  Слишком мало предложений для {parser}, пропускаем")
            continue
        nbrs = NearestNeighbors(n_neighbors=k, metric='precomputed').fit(combined_dist)
        distances, _ = nbrs.kneighbors(combined_dist)
        kth_dist = np.sort(distances[:, -1])
        eps = np.percentile(kth_dist, args.eps_quantile * 100)
        db = DBSCAN(eps=eps, min_samples=args.min_samples, metric='precomputed')
        labels = db.fit_predict(combined_dist)

        # Вычисляем метрики для каждого предложения
        # WL для всех пар
        wl_scores = new_wl_ker1(gold_graphs, parser_graphs, h=3)

        records = []
        for i, (sid, dataset) in enumerate(valid_sids):
            gold_deps = gold_deps_list[i]
            parser_deps = parser_deps_list[i]
            uas, las = compute_las_uas(gold_deps, parser_deps)
            crit = compute_structural_criteria(gold_deps, parser_deps)
            records.append({
                'sent_id': sid,
                'dataset': dataset,
                'cluster': labels[i],
                'UAS': uas,
                'LAS': las,
                'WL': wl_scores[i],
                'wrong_root': crit['wrong_root'],
                'parent_ratio': crit['parent_ratio'],
                'depth_diff': crit['depth_diff'],
                'branch_diff': crit['branch_diff'],
                'reversed': crit['reversed'],
                'depth_shift_ratio': crit['depth_shift_ratio'],
                'SimpleStruct': crit['SimpleStruct']
            })

        df_metrics = pd.DataFrame(records)
        # Сохраняем метрики и кластеры вместе
        metrics_file = os.path.join(args.out_dir, f'metrics_{parser}.csv')
        df_metrics.to_csv(metrics_file, index=False)

        # Для обратной совместимости сохраняем отдельный файл с кластерами (можно не сохранять, но оставим)
        cluster_file = os.path.join(args.out_dir, f'clusters_{parser}_pairwise.csv')
        df_metrics[['sent_id', 'dataset', 'cluster']].to_csv(cluster_file, index=False)

        results[parser] = {
            'n_clusters': len(set(labels)) - (1 if -1 in labels else 0),
            'noise_count': int(np.sum(labels == -1)),
            'total': len(labels),
            'eps': eps,
            'skipped': len(sids) - len(valid_sids)
        }
        print(f"  {parser}: кластеров {results[parser]['n_clusters']}, шума {results[parser]['noise_count']}, пропущено {results[parser]['skipped']}")

    summary_df = pd.DataFrame.from_dict(results, orient='index')
    summary_df.to_csv(os.path.join(args.out_dir, 'summary_pairwise.csv'))
    print("\nГотово.")


if __name__ == '__main__':
    main()

Overwriting cluster_pairwise_distance.py


In [ ]:
!conda run -n g2v37 python cluster_short_sentences.py \
    --max-length 20 \
    --eps-quantile 0.75 \
    --min-samples 5 \
    --out-dir /content/cluster_short_results


Обработка парсера natasha...
  natasha: кластеров 1, шума 784

Обработка парсера udpipe...
  udpipe: кластеров 1, шума 743

Обработка парсера spacy...
  spacy: кластеров 1, шума 745

Обработка парсера deeppavlov...
  deeppavlov: кластеров 4, шума 761

Обработка парсера stanza...
  stanza: кластеров 2, шума 768

Готово.



In [ ]:
!conda run -n g2v37 python cluster_pairwise_distance.py --max-length 20 --eps-quantile 0.70 --min-samples 10 --out-dir /content/cluster_pairwise_results


Обработка парсера natasha...
  natasha: кластеров 9, шума 1664, пропущено 0

Обработка парсера udpipe...
  udpipe: кластеров 14, шума 1797, пропущено 0

Обработка парсера spacy...
  spacy: кластеров 8, шума 1766, пропущено 0

Обработка парсера deeppavlov...
  deeppavlov: кластеров 21, шума 1922, пропущено 0

Обработка парсера stanza...
  stanza: кластеров 18, шума 1915, пропущено 0

Готово.



In [ ]:
%%writefile analyze_clusters.py
"""
Скрипт для анализа результатов кластеризации:
- загрузка CSV-файлов с кластерами
- вычисление статистики по кластерам (размер, датасеты, метрики, типы ошибок)
- визуализация кластеров в 2D (UMAP)
- генерация отчёта с интерпретацией кластеров
"""

import os
import sys
import pickle
import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, Counter
from pathlib import Path

# Для визуализации
try:
    import umap
    HAVE_UMAP = True
except ImportError:
    HAVE_UMAP = False
    from sklearn.manifold import TSNE

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Добавляем путь к модулям, если нужно
sys.path.append('/content/parser_stat')

from dep_metrics import find_root, structural_distortion_score, tree_depth_and_branching
from graph2vec_clustering import ExperimentConfig, deps_to_graph, fit_graph2vec


def load_pickle(path):
    with open(path, 'rb') as f:
        return pickle.load(f)


def remove_punct(deps):
    return {t: (h, r) for t, (h, r) in deps.items() if r != 'punct'}


def compute_structural_criteria(gold_deps, parser_deps):
    """Вычисляет критерии структурных ошибок для пары деревьев."""
    import networkx as nx
    gold_root = find_root(gold_deps)
    parser_root = find_root(parser_deps)
    gold_tokens = set(gold_deps.keys())
    parser_tokens = set(parser_deps.keys())
    common = gold_tokens & parser_tokens

    wrong_root = 1 if gold_root != parser_root else 0
    if not common:
        parent_ratio = 1.0
        depth_shift_ratio = 1.0
        reversed_flag = 0
    else:
        # parent_ratio
        changed_parent = sum(1 for t in common if gold_deps[t][0] != parser_deps[t][0])
        parent_ratio = changed_parent / len(common)

        # depth_shift_ratio
        def compute_depths(deps, root):
            G = nx.DiGraph()
            G.add_nodes_from(deps.keys())
            for t, (h, _) in deps.items():
                if h != (-1, -1):
                    G.add_edge(h, t)
            depths = {}
            if root in G:
                queue = [(root, 0)]
                visited = {root}
                while queue:
                    node, d = queue.pop(0)
                    depths[node] = d
                    for child in G.successors(node):
                        if child not in visited:
                            visited.add(child)
                            queue.append((child, d + 1))
            return depths
        gold_depths = compute_depths(gold_deps, gold_root)
        parser_depths = compute_depths(parser_deps, parser_root)
        depth_shift = sum(1 for t in common if abs(gold_depths.get(t, 0) - parser_depths.get(t, 0)) > 1)
        depth_shift_ratio = depth_shift / len(common)

        # reversed
        gold_edges = {(h, t) for t, (h, _) in gold_deps.items() if h != (-1, -1)}
        parser_edges = {(h, t) for t, (h, _) in parser_deps.items() if h != (-1, -1)}
        reversed_flag = 1 if any((v, u) in parser_edges for (u, v) in gold_edges) else 0

    gold_depth, gold_branch = tree_depth_and_branching(gold_deps, gold_root)
    parser_depth, parser_branch = tree_depth_and_branching(parser_deps, parser_root)
    depth_diff = abs(gold_depth - parser_depth) / max(gold_depth, parser_depth, 1)
    branch_diff = abs(gold_branch - parser_branch) / max(gold_branch, parser_branch, 1e-6)

    return {
        'wrong_root': wrong_root,
        'parent_ratio': parent_ratio,
        'depth_diff': depth_diff,
        'branch_diff': branch_diff,
        'reversed': reversed_flag,
        'depth_shift_ratio': depth_shift_ratio,
        'SimpleStruct': (parent_ratio + depth_diff + branch_diff + depth_shift_ratio + wrong_root + reversed_flag) / 6.0
    }


def main():
    parser = argparse.ArgumentParser(description='Анализ результатов кластеризации')
    parser.add_argument('--gold', type=str, default='/content/be_treebanks.pkl')
    parser.add_argument('--parser-file', type=str, default='/content/be_parser_res.pkl')
    parser.add_argument('--short-dir', type=str, default='/content/cluster_short_results',
                        help='Директория с результатами кластеризации коротких предложений')
    parser.add_argument('--pairwise-dir', type=str, default='/content/cluster_pairwise_results',
                        help='Директория с результатами pairwise кластеризации')
    parser.add_argument('--out-dir', type=str, default='/content/cluster_analysis')
    parser.add_argument('--max-plot-samples', type=int, default=2000, help='Максимальное число точек для 2D-проекции')
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)

    # Загрузка данных
    be_treebanks = load_pickle(args.gold)
    all_parsers = load_pickle(args.parser_file)
    parser_names = list(all_parsers.keys())
    treebank_names = list(be_treebanks.keys())

    # Загружаем результаты кластеризации
    # Для short – используем clusters_*_short.csv (там только кластеры)
    short_clusters = {}
    for parser in parser_names:
        fname = os.path.join(args.short_dir, f'clusters_{parser}_short.csv')
        if os.path.exists(fname):
            short_clusters[parser] = pd.read_csv(fname)
        else:
            print(f"Файл {fname} не найден")

    # Для pairwise – используем metrics_*_pairwise.csv (там есть кластеры + метрики)
    pairwise_metrics = {}
    for parser in parser_names:
        fname = os.path.join(args.pairwise_dir, f'metrics_{parser}.csv')
        if os.path.exists(fname):
            pairwise_metrics[parser] = pd.read_csv(fname)
        else:
            print(f"Файл {fname} не найден")

    # Функция для сбора метрик для каждого предложения (для short, если нужно)
    def collect_metrics(sent_id, dataset, parser):
        gold_deps = remove_punct(be_treebanks[dataset][sent_id])
        parser_deps = remove_punct(all_parsers[parser][dataset][sent_id])
        crit = compute_structural_criteria(gold_deps, parser_deps)
        return {
            'length': len(gold_deps),
            **crit
        }

    # Анализ для коротких предложений (short)
    print("\n=== АНАЛИЗ КЛАСТЕРИЗАЦИИ: SHORT ===")
    for parser, df in short_clusters.items():
        print(f"\nПарсер: {parser.upper()}")
        print(f"Всего предложений: {len(df)}")
        print(f"Кластеров: {df['cluster'].nunique() - (1 if -1 in df['cluster'].values else 0)}")
        print(f"Шум: {(df['cluster'] == -1).sum()}")

        cluster_stats = df.groupby('cluster').agg(
            count=('sent_id', 'count'),
            datasets=('dataset', lambda x: dict(Counter(x))),
        )
        print("\nСтатистика по кластерам (первые 10):")
        print(cluster_stats.head(10))
        summary_file = os.path.join(args.out_dir, f'cluster_summary_short_{parser}.csv')
        cluster_stats.to_csv(summary_file)
        print(f"Сводка сохранена в {summary_file}")

    # Анализ для pairwise (используем metrics_{parser}.csv)
    print("\n=== АНАЛИЗ КЛАСТЕРИЗАЦИИ: PAIRWISE ===")
    for parser, df in pairwise_metrics.items():
        print(f"\nПарсер: {parser.upper()}")
        print(f"Всего предложений: {len(df)}")
        print(f"Кластеров: {df['cluster'].nunique() - (1 if -1 in df['cluster'].values else 0)}")
        print(f"Шум: {(df['cluster'] == -1).sum()}")

        # Статистика по кластерам с метриками
        cluster_metrics = df.groupby('cluster').agg({
            'UAS': 'mean',
            'LAS': 'mean',
            'WL': 'mean',
            'wrong_root': 'mean',
            'parent_ratio': 'mean',
            'depth_diff': 'mean',
            'branch_diff': 'mean',
            'reversed': 'mean',
            'depth_shift_ratio': 'mean',
            'SimpleStruct': 'mean',
            'sent_id': 'count',
            'dataset': lambda x: dict(Counter(x))
        }).rename(columns={'sent_id': 'count'})
        print("\nСтатистика по кластерам (первые 10):")
        print(cluster_metrics.head(10))

        # Сохраняем
        summary_file = os.path.join(args.out_dir, f'cluster_summary_pairwise_{parser}.csv')
        cluster_metrics.to_csv(summary_file)
        print(f"Сводка сохранена в {summary_file}")

        # Дополнительно: средние значения для шума и не-шума
        noise_df = df[df['cluster'] == -1]
        non_noise_df = df[df['cluster'] != -1]
        if len(noise_df) > 0:
            print(f"\nШум: средние метрики")
            print(noise_df[['UAS', 'LAS', 'WL', 'SimpleStruct']].mean())
        if len(non_noise_df) > 0:
            print(f"\nНе-шум: средние метрики")
            print(non_noise_df[['UAS', 'LAS', 'WL', 'SimpleStruct']].mean())

    print("\nАнализ завершён.")


if __name__ == '__main__':
    main()

Overwriting analyze_clusters.py


In [ ]:
!conda run -n g2v37 python -m pip install seaborn

In [ ]:
!conda run -n g2v37 python analyze_clusters.py --short-dir /content/cluster_short_results --pairwise-dir /content/cluster_pairwise_results --out-dir /content/cluster_analysis


=== АНАЛИЗ КЛАСТЕРИЗАЦИИ: SHORT ===

Парсер: NATASHA
Всего предложений: 9239
Кластеров: 1
Шум: 784

Статистика по кластерам (первые 10):
         count                                           datasets
cluster                                                          
-1         784  {'taiga': 81, 'poetry': 59, 'gsd': 49, 'pud': ...
 0        8455  {'taiga': 740, 'poetry': 566, 'gsd': 394, 'pud...
Сводка сохранена в /content/cluster_analysis/cluster_summary_short_natasha.csv

Парсер: UDPIPE
Всего предложений: 9239
Кластеров: 1
Шум: 743

Статистика по кластерам (первые 10):
         count                                           datasets
cluster                                                          
-1         743  {'taiga': 70, 'poetry': 53, 'gsd': 58, 'pud': ...
 0        8496  {'taiga': 751, 'poetry': 572, 'gsd': 385, 'pud...
Сводка сохранена в /content/cluster_analysis/cluster_summary_short_udpipe.csv

Парсер: SPACY
Всего предложений: 9239
Кластеров: 1
Шум: 745

Статистика по к